# Baseline Ottobiss + готовые Z / R / ZR: финальный mIoU

**Никакого обучения B0/B5/AE/CatBoost.** Используются готовые чекпоинты пользователя и scorer из `SAE/INTEGRATION.md`. Подгоняются только банк kNN и чистые пороги `flag`, как в baseline.

Пять вариантов с одним и тем же правилом `Router` Ottobiss:

| Метод | flag (больше — необычный кадр) | priority |
|---|---|---|
| entropy | нет | энтропия |
| knn_entropy | kNN | энтропия |
| Z_entropy | среднее готовой карты Z | энтропия |
| R_entropy | среднее готовой карты R | энтропия |
| ZR_entropy | среднее готовой карты ZR | энтропия |

Карта — предсказанная доля ошибок B0 в патчах16×16. Её усреднение по кадру — **наша проверяемая связь со baseline**, не доказанный прогноз пользы B5. Правило выдаёт light / ens@0.5 / ens@0.75 / heavy. Бюджет0.5 — площадь входов B5 в среднем, **не время**. Фактические расходы сравниваем отдельно.

Источники:

- baseline `main` commit `8eb70cae33a3875a2588a350719ac6002d77347f`, `router.py` и kNN/entropy скопированы без изменений;
- scorer и все перечисленные в инструкции функции SAE скопированы, изменены только пути импортов в самостоятельном пакете;
- [INTEGRATION.md](https://github.com/Ottobiss/error-geometry-routing/blob/SAE/INTEGRATION.md), [baseline](https://github.com/Ottobiss/error-geometry-routing/tree/8eb70cae33a3875a2588a350719ac6002d77347f).

**Пересечение P0:** готовые CatBoost обучены на300 P0, включая141 кадр из тестовых городов Ottobiss. Удаляем все P0 из теста Cityscapes для всех методов. Остаются **92 Cityscapes +406 ACDC =498 кадров**. Подгонка банка/порогов —267 Frankfurt, не входит в тест. Не используем исходный пул639 как независимый тест готового Z/R/ZR.

**Единый инференс:** B0 и B5 fp32 у всех вариантов; Cityscapes1024×512, ACDC1024×576 по SAE guide. Базовый main использовал fp16 и ACDC1024×512. Здесь вероятности поднимаются до рабочего размера до ансамблирования. Поэтому это контролируемый повтор правила baseline, **не воспроизведение его опубликованных чисел**. Все пять методов и полный B5 измеряются в одном новом протоколе.

Этапы: установка → чекпоинты/данные → acceptance (`max_error <1e-6`) → fit без обучения сетей → кэш всех уровней → mIoU/CI → живое время → экспорт.

Главное сравнение заранее: ZR_entropy − knn_entropy, порядок блоками, общий пул. Z/R и перестановки — вторичные. Все варианты публикуем; лучший по ACDC не выбираем. Данные уже просматривались: выводы исследовательские. Bootstrap по кадрам и последовательностям условен на реализованных решениях потока, без повторного роутинга внутри bootstrap; без поправок за множественные сравнения.

Метрики: общий mIoU из суммы матриц ошибок (не среднее mIoU кадров), по условиям/ACDC/пулу; затраты B5 по площади; парные CI; среднее время/p95 на фиксированных40 кадрах. Не объявляем победу при равном времени, если время разошлось. Непрерывный поток рассматриваем блоками и в5 фиксированных перестановках.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json, time, zipfile
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/ottobiss_zrz_integration_v1')
OUT = Path('/content/drive/MyDrive/ottobiss_zrz_integration_v1/results')
EXP = ROOT / 'experiments'
ART = ROOT / 'artifacts/results/ae'
CS = ROOT / 'data/cityscapes'
ACDC = ROOT / 'data/acdc'
for p in [ROOT, OUT, EXP, ART, CS, ACDC]: p.mkdir(parents=True, exist_ok=True)
print('Результаты:', OUT)
print('Локально свободно:', round(shutil.disk_usage(ROOT).free/2**30, 1), 'ГиБ')


## 1. Изолированное окружение по версиям `INTEGRATION.md`

Системный PyTorch и прежние ноутбуки не заменяем. Здесь Python3.14 и версии, заявленные автором инструкции: torch2.14.1+cu130, transformers5.20.2, catboost1.2.10, pandas3.0.6, pyarrow26.0.0. Установка загружает отдельный Python/PyTorch и требует места/времени. Эти версии не устанавливались и не проверялись на GPU в среде автора этого ноутбука. Если нужный wheel недоступен или установка падает — остановись и пришли лог, не подменяй версии произвольно и не ослабляй acceptance.

Нужен GPU-runtime. Во время долгой установки используется вывод установщика; новую копию установки параллельно не запускай.

In [ ]:
os.environ.update({
    'UV_PYTHON_INSTALL_DIR': str(ROOT/'python'),
    'UV_CACHE_DIR': str(ROOT/'uv_cache'),
    'PIP_NO_CACHE_DIR': '1',
    'USE_TF': '0',
    'OMP_NUM_THREADS': '4',
    'OPENBLAS_NUM_THREADS': '4',
    'MKL_NUM_THREADS': '4',
})
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
assert UV, 'Не найден uv'
VENV = ROOT / 'venv_reference'
PYTHON = VENV / 'bin/python'
if not PYTHON.is_file():
    subprocess.run([UV, 'venv', '--python', '3.14', str(VENV)], check=True)
ready = VENV/'ready.json'
if not ready.is_file():
    subprocess.run([UV, 'pip', 'install', '--python', str(PYTHON),
        'torch==2.14.1', '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
    subprocess.run([UV, 'pip', 'install', '--python', str(PYTHON),
        'transformers==5.20.2', 'catboost==1.2.10', 'pandas==3.0.6',
        'pyarrow==26.0.0', 'numpy', 'Pillow', 'gdown>=6.3.0'], check=True)

check = """
import sys, torch, transformers, catboost, pandas, pyarrow
from transformers import SegformerForSemanticSegmentation
assert sys.version_info[:2]==(3,14)
assert torch.__version__.split('+')[0]=='2.14.1'
assert transformers.__version__=='5.20.2'
assert catboost.__version__=='1.2.10'
assert pandas.__version__=='3.0.6'
assert pyarrow.__version__=='26.0.0'
assert torch.cuda.is_available(), 'Включи GPU'
print(sys.version,torch.__version__,transformers.__version__,torch.cuda.get_device_name(0))
"""
subprocess.run([str(PYTHON), '-c', check], check=True)
ready.write_text(json.dumps({'reference_versions_checked': True}))


In [ ]:
SOURCES = {'experiment.py': '"""Z/R/ZR scorer integration: exact SAE scorer, unchanged Ottobiss Router rule.\nControlled FP32 rerun; working sizes follow SAE, not published main numbers.\n"""\nimport os\nos.environ[\'USE_TF\']=\'0\'\nimport argparse,json,hashlib,time,copy\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom scorer import ErrorScorer,F0\nfrom sae_lib import load_model,output_features\nfrom router import Router,LEVELS\nfrom features.knn import KNN\nfrom features.entropy import Entropy\n\nROOT=Path(os.environ[\'ZR_ROOT\']);OUT=Path(os.environ[\'ZR_OUT\']);ART=Path(os.environ[\'ZR_ART\']);CS=Path(os.environ[\'ZR_CS\']);ACDC=Path(os.environ[\'ZR_ACDC\']);CACHE=ROOT/\'cache\'\nKINDS=[\'Z\',\'R\',\'ZR\'];METHODS=[\'entropy\',\'knn_entropy\',\'Z_entropy\',\'R_entropy\',\'ZR_entropy\'];MODES=[\'light\',\'ens@0.5\',\'ens@0.75\',\'heavy\'];DOMAINS=[\'cs_test\',\'fog\',\'night\',\'rain\',\'snow\'];BUDGET=.5;QUANTILE=.95;WINDOW=50\nLUT=np.full(256,255,np.uint8)\nfor a,b in {7:0,8:1,11:2,12:3,13:4,17:5,19:6,20:7,21:8,22:9,23:10,24:11,25:12,26:13,27:14,28:15,31:16,32:17,33:18}.items():LUT[a]=b\n\ndef sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef write(p,v):\n p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(p.suffix+\'.part\');t.write_text(json.dumps(v,indent=2));t.replace(p)\ndef fixed(p,v):\n if p.exists():assert json.loads(p.read_text())==v,f\'Inputs/code changed: {p}. Use a new output directory.\'\n else:write(p,v)\ndef save(p,**kw):\n p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(\'.npz.part\')\n with t.open(\'wb\') as f:np.savez_compressed(f,**kw)\n t.replace(p)\ndef conf(gt,p):\n ok=gt!=255;return np.bincount(gt[ok].astype(np.int64)*19+p[ok].astype(np.int64),minlength=361).reshape(19,19)\ndef miou(c):\n c=c.astype(float);d=np.diag(c);u=c.sum(0)+c.sum(1)-d;return float(100*(d[u>0]/u[u>0]).mean())\ndef csv(name,rs):\n pd.DataFrame(rs).to_csv(OUT/name,index=False)\n\ndef inventory():\n fit=[];test=[]\n for city in [\'frankfurt\',\'lindau\',\'munster\']:\n  files=sorted((CS/\'leftImg8bit/val\'/city).glob(\'*_leftImg8bit.png\'));assert files,f\'No Cityscapes {city}\'\n  for p in files:\n   name=p.name[:-len(\'_leftImg8bit.png\')];gt=CS/\'gtFine/val\'/city/(name+\'_gtFine_labelIds.png\');assert gt.is_file(),str(gt)\n   r=dict(name=name,domain=\'cs_fit\' if city==\'frankfurt\' else \'cs_test\',sequence=\'_\'.join(name.split(\'_\')[:2]),rgb=str(p),gt=str(gt),size=[1024,512],cityscapes=True)\n   (fit if city==\'frankfurt\' else test).append(r)\n p0=set(Path(__file__).with_name(\'p0_ids.txt\').read_text().split());assert len(p0)==300\n test=[r for r in test if r[\'name\'] not in p0]\n assert not any(r[\'name\'] in p0 for r in test)\n for d in [\'fog\',\'night\',\'rain\',\'snow\']:\n  files=sorted((ACDC/\'rgb_anon\'/d/\'val\').rglob(\'*_rgb_anon.png\'));assert files,f\'No ACDC {d}\'\n  for p in files:\n   stem=p.name[:-len(\'_rgb_anon.png\')];gt=ACDC/\'gt\'/d/\'val\'/p.parent.name/(stem+\'_gt_labelTrainIds.png\');assert gt.is_file(),str(gt)\n   test.append(dict(name=d+\'_\'+stem,domain=d,sequence=d+\'/\'+p.parent.name,rgb=str(p),gt=str(gt),size=[1024,576],cityscapes=False))\n assert len({r[\'name\'] for r in fit+test})==len(fit)+len(test)\n assert len(fit)==267 and len(test)==498,(len(fit),len(test))\n return fit,test\n\ndef setup():\n OUT.mkdir(parents=True,exist_ok=True);CACHE.mkdir(parents=True,exist_ok=True)\n for k in KINDS:\n  for n in [\'model.cbm\',\'scores.parquet\']:assert (ART/\'runs\'/(k+\'_ae_b8_catboost\')/n).is_file(),f\'Missing {k}/{n}\'\n weights=ART/\'ckpt/ae_b8\'\n assert len(list(weights.glob(\'s*b*.pt\')))==8\n fit,test=inventory()\n fixed(OUT/\'protocol.json\',dict(schema=\'ottobiss-SAE-Z-R-ZR-fixed-v1\',baseline_commit=\'8eb70cae33a3875a2588a350719ac6002d77347f\',methods=METHODS,budget=BUDGET,quantile=QUANTILE,window=WINDOW,flag=\'mean full-frame scorer map; fixed clean-data q95\',priority=\'native B0 logit entropy as baseline\',B0=\'fp32 exact SAE SegModel\',B5=\'fp32 common to all methods\',sizes=\'CS 1024x512; ACDC 1024x576 per SAE guide\',inference=\'full working-size probabilities before ensemble; controlled rerun, not published main values\',cost=\'scale squared for B5 only; not latency or total FLOPs\',primary=\'ZR_entropy minus knn_entropy, blocked stream\',secondary=\'Z/R vs knn; all AE variants vs entropy; 5 fixed shuffled orders\',selection=\'no best-method selection on evaluation; report all\',bootstrap=\'2000 sequence and frame resamples of fixed predictions, conditional on realized routing; stream not rerouted inside bootstrap\',status=\'previously reused CS/ACDC exploratory; exclude all SAE P0 from evaluation; 92 CS + 406 ACDC; unadjusted CIs\',source={p.name:sha(p) for p in Path(__file__).parent.iterdir() if p.suffix in [\'.py\',\'.txt\']},features={p.name:sha(p) for p in Path(__file__).parent.joinpath(\'features\').glob(\'*.py\')},weights={b.name:sha(b) for b in weights.glob(\'*.pt\')},models={k:sha(ART/\'runs\'/(k+\'_ae_b8_catboost\')/\'model.cbm\') for k in KINDS},references={k:sha(ART/\'runs\'/(k+\'_ae_b8_catboost\')/\'scores.parquet\') for k in KINDS},runtime=dict(torch=str(torch.__version__),transformers=__import__(\'transformers\').__version__,catboost=__import__(\'catboost\').__version__,gpu=torch.cuda.get_device_name(0),tf32_matmul=torch.backends.cuda.matmul.allow_tf32,tf32_cudnn=torch.backends.cudnn.allow_tf32,deterministic=torch.backends.cudnn.deterministic)))\n fixed(OUT/\'manifest.json\',dict(fit=fit,test=test));return fit,test\n\ndef small(r):\n with Image.open(r[\'rgb\']) as im:a=np.array(im.convert(\'RGB\').resize(tuple(r[\'size\']),Image.Resampling.BILINEAR),copy=True)\n return torch.from_numpy(a).permute(2,0,1)\ndef ground(r):\n with Image.open(r[\'gt\']) as im:g=np.array(im.resize(tuple(r[\'size\']),Image.Resampling.NEAREST))\n if r[\'cityscapes\']:g=LUT[g]\n assert set(np.unique(g))<=set(range(19))|{255};return g\n\nclass Engine:\n def __init__(self):\n  self.b0=load_model(\'b0\').requires_grad_(False);self.b5=load_model(\'b5\').requires_grad_(False)\n  self.scorers={k:ErrorScorer(k,root=ART,b0=self.b0) for k in KINDS}\n  for scorer in self.scorers.values():\n   for enc in scorer.enc.values():enc.requires_grad_(False)\n  self.knn=KNN().cuda();self.entropy=Entropy().cuda()\n  fixed(OUT/\'model_revisions.json\',dict(b0=getattr(self.b0.net.config,\'_commit_hash\',None),b5=getattr(self.b5.net.config,\'_commit_hash\',None)))\n def native(self):return SimpleNamespace(logits=self.b0.features[\'logits\'][None],taps={\'decoder\':self.b0.features[\'C\']})\n @torch.inference_mode()\n def forward(self,img,kind=None):\n  w,h=img.shape[-1],img.shape[-2]\n  if kind is None:\n   probs=self.b0(img[None].cuda(),(w,h)).softmax(1)[0];return probs,{},float(self.entropy(self.native()))\n  result=self.scorers[kind](img);return result[\'probs\'],result,float(self.entropy(self.native()))\n @torch.inference_mode()\n def all_scores(self,img):\n  probs,out,ent=self.forward(img,\'ZR\')\n  f0=output_features(probs,out[\'pred\']);parts={\'F0\':pd.DataFrame({c:f0[c] for c in F0}),**{k:pd.DataFrame(out[k]) for k in [\'N\',\'Z\',\'R\']}}\n  maps={\'ZR\':out[\'score\']}\n  from catboost import Pool\n  for k in [\'Z\',\'R\']:\n   scorer=self.scorers[k];x=pd.concat([parts[p] for p in scorer.parts],axis=1);x.columns=scorer.model.feature_names_;x[\'cls\']=x[\'cls\'].astype(np.int32);score=scorer.model.predict(Pool(x,cat_features=[\'cls\']),prediction_type=\'Probability\')[:,1];maps[k]=score.reshape(out[\'score\'].shape).astype(np.float32)\n  knn=float(self.knn(self.native())) if self.knn.bank.numel() else None\n  return probs,maps,ent,knn\n @torch.inference_mode()\n def heavy_probs(self,img,scale):\n  h,w=img.shape[-2:];x=img[None].cuda().float()\n  if scale!=1.:x=F.interpolate(x,scale_factor=scale,mode=\'bilinear\',align_corners=False,antialias=True)\n  return self.b5(x,(w,h)).softmax(1)[0]\n @torch.inference_mode()\n def predict(self,img,probs,level):\n  if not level.scale:return probs.argmax(0).byte().cpu().numpy()\n  heavy=self.heavy_probs(img,level.scale)\n  return ((probs+heavy)/2 if level.ensemble else heavy).argmax(0).byte().cpu().numpy()\n\n@torch.inference_mode()\ndef acceptance():\n fit,test=setup();engine=Engine();refs={k:pd.read_parquet(ART/\'runs\'/(k+\'_ae_b8_catboost\')/\'scores.parquet\') for k in KINDS};checks=[]\n selected=[next(r for r in fit if r[\'name\']==\'frankfurt_000000_000576\')]+[next(r for r in test if r[\'domain\']==d) for d in [\'fog\',\'night\',\'rain\',\'snow\']]\n # Include author\'s published ACDC acceptance frame if present.\n selected += [r for r in test if r[\'name\']==\'night_GOPR0351_frame_000033\' and r not in selected]\n for r in selected:\n  img=small(r);p,maps,_,_=engine.all_scores(img)\n  plain=engine.b0(img[None].cuda(),tuple(r[\'size\'])).softmax(1)[0];assert torch.equal(p,plain),\'Scorer hooks altered B0 logits/probabilities\'\n  for k in KINDS:\n   ref=refs[k];ref=ref[(ref.dataset==(\'cityscapes\' if r[\'cityscapes\'] else \'acdc\'))&(ref.image_id==r[\'name\'])];assert len(ref)>0,r[\'name\']\n   d=np.abs(maps[k].ravel()[ref.patch.to_numpy().astype(int)]-ref.score.to_numpy());checks.append(dict(name=r[\'name\'],method=k,patches=len(ref),max_error=float(d.max()),mean_error=float(d.mean()),p99_error=float(np.quantile(d,.99))))\n  print(\'Acceptance\',r[\'name\'],flush=True)\n write(OUT/\'acceptance.json\',checks);passed=all(r[\'max_error\']<1e-6 for r in checks);write(OUT/\'acceptance_status.json\',dict(passed=passed));print(pd.DataFrame(checks).to_string(index=False),flush=True)\n assert passed,\'Acceptance failed: do not relax tolerance or continue. Reports saved in acceptance.json.\'\n\ndef require_acceptance():assert json.loads((OUT/\'acceptance_status.json\').read_text())[\'passed\'],\'Run acceptance first\'\n\n@torch.inference_mode()\ndef fit():\n train,test=setup();require_acceptance();engine=Engine()\n # Exact baseline KNN bank sampling, fitted only on clean Frankfurt frames.\n for i,r in enumerate(train):\n  engine.forward(small(r));engine.knn.update(engine.native())\n  if (i+1)%20==0:print(\'KNN bank\',i+1,\'/\',len(train),flush=True)\n engine.knn.finish();scores=[]\n for i,r in enumerate(train):\n  _,maps,ent,knn=engine.all_scores(small(r));scores.append([knn]+[float(maps[k].mean()) for k in KINDS])\n  if (i+1)%20==0:print(\'Fit flags\',i+1,\'/\',len(train),flush=True)\n thresholds={method:float(np.quantile(np.array(scores)[:,j],QUANTILE)) for j,method in enumerate(METHODS[1:])}\n value=dict(bank=engine.knn.bank.cpu(),thresholds=thresholds,protocol_sha=sha(OUT/\'protocol.json\'))\n dest=OUT/\'state.pt\';tmp=dest.with_suffix(\'.pt.part\');torch.save(value,tmp);tmp.replace(dest);save(OUT/\'fit_scores.npz\',scores=np.array(scores),names=np.array([r[\'name\'] for r in train]));write(OUT/\'thresholds.json\',thresholds);print(thresholds,flush=True)\n\ndef load_state(engine):\n state=torch.load(OUT/\'state.pt\',map_location=\'cpu\',weights_only=True);assert state[\'protocol_sha\']==sha(OUT/\'protocol.json\');engine.knn.bank=state[\'bank\'].cuda();return state\n\ndef routers(state):\n result={}\n for method in METHODS:\n  r=Router(BUDGET,QUANTILE,WINDOW)\n  if method!=\'entropy\':r.threshold=state[\'thresholds\'][method]\n  result[method]=r\n return result\n\ndef flag(scores,method):return None if method==\'entropy\' else float(scores[METHODS.index(method)])\n\n@torch.inference_mode()\ndef cache():\n train,test=setup();require_acceptance();engine=Engine();load_state(engine);start=time.monotonic()\n for i,r in enumerate(test):\n  dest=CACHE/(r[\'name\']+\'.npz\')\n  if dest.exists():continue\n  img=small(r);probs,maps,ent,knn=engine.all_scores(img);g=ground(r);cs=[conf(g,probs.argmax(0).byte().cpu().numpy())]\n  for level in MODES[1:]:cs.append(conf(g,engine.predict(img,probs,LEVELS[level])))\n  save(dest,confs=np.stack(cs),scores=np.array([ent,knn]+[float(maps[k].mean()) for k in KINDS]))\n  if (i+1)%5==0 or i+1==len(test):print(\'Cache\',i+1,\'/\',len(test),\'elapsed\',round((time.monotonic()-start)/60,1),\'min\',flush=True)\n write(OUT/\'cache_complete.json\',dict(complete=True,frames=len(test)));analyze()\n\ndef bootstrap(a,b,groups):\n unique=np.unique(groups);aa=np.stack([a[groups==g].sum(0) for g in unique]).astype(float);bb=np.stack([b[groups==g].sum(0) for g in unique]).astype(float);rng=np.random.default_rng(0)\n weights=rng.multinomial(len(unique),np.ones(len(unique))/len(unique),size=2000)\n def many(c):\n  c=c.reshape(-1,19,19);d=np.diagonal(c,axis1=1,axis2=2);u=c.sum(1)+c.sum(2)-d\n  return 100*np.nanmean(np.divide(d,u,out=np.full_like(d,np.nan),where=u>0),axis=1)\n differences=many(weights@aa.reshape(len(unique),-1))-many(weights@bb.reshape(len(unique),-1))\n return tuple(float(x) for x in np.percentile(differences,[2.5,97.5]))\n\ndef replay(test,state,order):\n rs=routers(state);cs=np.zeros((len(test),len(METHODS),19,19),np.int64);cost=np.zeros((len(test),len(METHODS)));levels=np.zeros((len(test),len(METHODS)),int)\n for i in order:\n  with np.load(CACHE/(test[i][\'name\']+\'.npz\'),allow_pickle=False) as z:\n   for j,m in enumerate(METHODS):\n    level=rs[m].decide(flag(z[\'scores\'],m),float(z[\'scores\'][0]));index=MODES.index(level.name);levels[i,j]=index;cost[i,j]=level.cost;cs[i,j]=z[\'confs\'][index]\n return cs,cost,levels\n\ndef analyze():\n train,test=setup();state=torch.load(OUT/\'state.pt\',map_location=\'cpu\',weights_only=True);summary=[];boots=[];names=np.array([r[\'name\'] for r in test]);groups=np.array([r[\'sequence\'] for r in test]);domains=np.array([r[\'domain\'] for r in test]);base=[];full=[]\n for r in test:\n  with np.load(CACHE/(r[\'name\']+\'.npz\'),allow_pickle=False) as z:base.append(z[\'confs\'][0]);full.append(z[\'confs\'][-1])\n base=np.stack(base);full=np.stack(full)\n for seed in [None,0,1,2,3,4]:\n  order=np.arange(len(test)) if seed is None else np.random.default_rng(seed).permutation(len(test));label=\'blocked\' if seed is None else \'shuffle\'+str(seed);cs,cost,levels=replay(test,state,order);save(OUT/(label+\'_confusions.npz\'),confs=cs,cost=cost,levels=levels,names=names,groups=groups,domains=domains,methods=np.array(METHODS),order=order)\n  for domain in DOMAINS+[\'acdc_all\',\'pool\']:\n   ix=np.ones(len(test),bool) if domain==\'pool\' else domains!=\'cs_test\' if domain==\'acdc_all\' else domains==domain\n   for j,m in enumerate(METHODS):summary.append(dict(order=label,domain=domain,method=m,miou=miou(cs[ix,j].sum(0)),frames=int(ix.sum()),mean_b5_area_cost=float(cost[ix,j].mean())))\n   for name,c in [(\'b0\',base),(\'b5_full\',full)]:summary.append(dict(order=label,domain=domain,method=name,miou=miou(c[ix].sum(0)),frames=int(ix.sum()),mean_b5_area_cost=0 if name==\'b0\' else 1))\n   pairs=[(\'ZR_entropy\',\'knn_entropy\'),(\'Z_entropy\',\'knn_entropy\'),(\'R_entropy\',\'knn_entropy\'),(\'knn_entropy\',\'entropy\')]+[(k+\'_entropy\',\'entropy\') for k in KINDS]\n   for a,b in pairs:\n    aa=cs[ix,METHODS.index(a)];bb=cs[ix,METHODS.index(b)]\n    # Group CI for every order; frame CI for blocked only to limit CPU work.\n    for unit,gg in [(\'sequence\',groups[ix])]+([(\'frame\',names[ix])] if seed is None else []):\n     lo,hi=bootstrap(aa,bb,gg);boots.append(dict(order=label,domain=domain,comparison=a+\' minus \'+b,unit=unit,groups=len(np.unique(gg)),delta_pp=miou(aa.sum(0))-miou(bb.sum(0)),ci_low=lo,ci_high=hi,primary=label==\'blocked\' and domain==\'pool\' and a==\'ZR_entropy\' and b==\'knn_entropy\',scope=\'exploratory; unadjusted; fixed realized stream decisions\'))\n csv(\'final_miou.csv\',summary);csv(\'bootstrap.csv\',boots);print(pd.DataFrame(summary).query("order==\'blocked\' and domain in [\'pool\',\'acdc_all\']").to_string(index=False));print(pd.DataFrame(boots).query("order==\'blocked\' and domain==\'pool\' and unit==\'sequence\'").to_string(index=False));write(OUT/\'quality_complete.json\',dict(complete=True,frames=len(test),orders=6,speed_claim=False))\n\n@torch.inference_mode()\ndef timing():\n train,test=setup();require_acceptance();engine=Engine();state=load_state(engine);rs=routers(state);selected=[]\n for d in DOMAINS:\n  ix=[i for i,r in enumerate(test) if r[\'domain\']==d];selected += [ix[j] for j in np.linspace(0,len(ix)-1,min(8,len(ix))).astype(int)]\n # Restore the exact preceding stream state for each timed frame, outside the clock.\n snapshots={};decisions={}\n for i,r in enumerate(test):\n  with np.load(CACHE/(r[\'name\']+\'.npz\'),allow_pickle=False) as z:\n   for method in METHODS:\n    if i in selected:snapshots[i,method]=copy.deepcopy(rs[method])\n    decisions[i,method]=rs[method].decide(flag(z[\'scores\'],method),float(z[\'scores\'][0])).name\n def call(r,method,router):\n  torch.cuda.synchronize();start=time.perf_counter();img=small(r)\n  if method==\'b5_full\':answer=engine.heavy_probs(img,1.).argmax(0).byte().cpu().numpy();level=\'heavy\'\n  else:\n   kind=method.split(\'_\')[0] if method.split(\'_\')[0] in KINDS else None\n   probs,out,ent=engine.forward(img,kind)\n   fl=float(out[\'score\'].mean()) if kind else float(engine.knn(engine.native())) if method==\'knn_entropy\' else None\n   decision=router.decide(fl,ent);answer=engine.predict(img,probs,decision);level=decision.name\n  torch.cuda.synchronize();return answer,level,(time.perf_counter()-start)*1000\n for method in METHODS+[\'b5_full\']:call(test[selected[0]],method,copy.deepcopy(snapshots.get((selected[0],method))))\n for pos,i in enumerate(selected):\n  dest=OUT/\'timing_records\'/(test[i][\'name\']+\'.json\')\n  if dest.exists():continue\n  record={m:[] for m in METHODS+[\'b5_full\']};tasks=[(m,rep) for m in record for rep in range(2)];np.random.default_rng(i).shuffle(tasks)\n  for method,rep in tasks:\n   answer,level,ms=call(test[i],method,copy.deepcopy(snapshots.get((i,method))))\n   with np.load(CACHE/(test[i][\'name\']+\'.npz\'),allow_pickle=False) as z:expected=z[\'confs\'][MODES.index(level)]\n   assert level==(\'heavy\' if method==\'b5_full\' else decisions[i,method]),f\'Live level mismatch: {test[i]["name"]} {method}\'\n   assert np.array_equal(conf(ground(test[i]),answer),expected),f\'Live/cache quality mismatch: {test[i]["name"]} {method}\'\n   record[method].append(float(ms))\n  write(dest,record);print(\'Timing\',pos+1,\'/\',len(selected),flush=True)\n reports=[]\n for method in METHODS+[\'b5_full\']:\n  times=np.array([json.loads((OUT/\'timing_records\'/(test[i][\'name\']+\'.json\')).read_text())[method] for i in selected]).ravel();reports.append(dict(method=method,mean_ms=float(times.mean()),p95_ms=float(np.percentile(times,95)),frames=len(selected),repeats=2,scope=\'fixed 40-frame subset, blocked-stream states, CPU decode to CPU mask, excludes GT; not all498\'))\n csv(\'timing.csv\',reports);print(pd.DataFrame(reports).to_string(index=False))\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'preflight\',\'acceptance\',\'fit\',\'cache\',\'analyze\',\'timing\']);args=p.parse_args();torch.set_num_threads(4);assert torch.cuda.is_available(),\'Enable GPU\'\n if args.stage==\'preflight\':f,t=setup();print(\'Inputs OK:\',len(f),\'fit frames,\',len(t),\'evaluation frames\')\n else:globals()[args.stage]()\n', 'scorer.py': '"""Patch error score of SegFormer-B0 from bottleneck-AE features (R, Z, ZR, F3F0NR)."""\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom catboost import CatBoost, Pool\n\nfrom sae_lib import (BLOCKS, PATCH, MixFFNTap, frame_features, load_encoder, load_model, output_features,\n                          pool_to_patches)\n\nF0 = ["ent_mean", "ent_max", "ent_p90", "msp_mean", "msp_min", "margin_mean", "margin_min",\n      "boundary_frac", "row", "col", "cls"]\nPARTS = {"R": ("F0", "N", "R"), "Z": ("F0", "Z"), "ZR": ("F0", "N", "Z", "R"), "F3F0NR": ("raw", "F0", "N", "R")}\n\n\nclass ErrorScorer:\n    """features: "R", "Z", "ZR" or "F3F0NR"; root: the results/ae folder of the SAE project, with\n    ckpt/ae_b8/<block>.pt and runs/<features>_ae_b8_catboost/model.cbm."""\n\n    def __init__(self, features, root="results/ae", device="cuda", b0=None):\n        root = Path(root)\n        self.parts = PARTS[features]\n        self.b0 = b0 if b0 is not None else load_model("b0", device)\n        self.enc = {b: load_encoder(root / "ckpt" / "ae_b8" / f"{b}.pt", device)[0] for b in BLOCKS}\n        self.norms = {b: (e.mean, e.scale.item()) for b, e in self.enc.items()}\n        self.model = CatBoost()\n        self.model.load_model(str(root / "runs" / f"{features}_ae_b8_catboost" / "model.cbm"))\n\n    @torch.inference_mode()\n    def __call__(self, small):\n        """small: (3, H, W) uint8 RGB frame at the working resolution, H and W multiples of 16.\n        Returns score (Hp, Wp), B0 pred (H, W), probs (19, H, W) and the per-patch features N, Z, R\n        (and raw for F3F0NR)."""\n        H, W = small.shape[1:]\n        hw = (H // PATCH, W // PATCH)\n        with MixFFNTap(self.b0) as tap:\n            probs = self.b0(small[None].to(self.b0.mean.device), (W, H)).softmax(1)[0]\n            pred = probs.argmax(0)\n            enc_out, N = frame_features(tap.acts, {"ae": self.enc}, self.norms, hw)\n            if "raw" in self.parts:      # max over the patch positions, stored as fp16 (as in training)\n                raw = np.concatenate([pool_to_patches(tap.acts[b].float(), b, hw).half().cpu().numpy()\n                                      for b in BLOCKS], axis=1).astype(np.float32)\n        Z, R = enc_out["ae"]\n        f0 = output_features(probs, pred)\n        feats = {"N": N, "Z": Z, "R": R, **({"raw": raw} if "raw" in self.parts else {})}\n        cols = {"F0": pd.DataFrame({c: f0[c] for c in F0}), **{k: pd.DataFrame(v) for k, v in feats.items()}}\n        X = pd.concat([cols[p] for p in self.parts], axis=1)\n        X.columns = self.model.feature_names_\n        X["cls"] = X["cls"].astype(np.int32)\n        score = self.model.predict(Pool(X, cat_features=["cls"]), prediction_type="Probability")[:, 1]\n        return {"score": score.reshape(hw).astype(np.float32), "pred": pred, "probs": probs, **feats}\n', 'sae_lib.py': 'import numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom transformers import SegformerForSemanticSegmentation\nNC = 19\n\n\nCHECKPOINTS = {\n    "b0": "nvidia/segformer-b0-finetuned-cityscapes-512-1024",\n    "b2": "nvidia/segformer-b2-finetuned-cityscapes-1024-1024",\n    "b5": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",\n}\n\nFULL_RES = {"b2", "b5"}\n\ndef upsample_logits(logits, size):\n    """Логиты (19, h, w) → (19, H, W) на рабочем разрешении size = (W, H), билинейно."""\n    x = torch.as_tensor(logits, device="cuda").float()[None]\n    return F.interpolate(x, size=(size[1], size[0]), mode="bilinear", align_corners=False)[0]\n\nclass SegModel(nn.Module):\n    """SegFormer с предобработкой. forward(image uint8 (B, 3, H, W), size) → логиты (B, 19, h, w)\n    на рабочем разрешении size = (W, h). У лёгкой модели после каждого прохода в self.features —\n    внутренние признаки (у тяжёлых хуков нет, чтобы не держать их тензоры в памяти GPU):\n\n    C      — вход decode_head.classifier: (1, 256, h/4, w/4) (после linear_fuse → BN → ReLU;\n             dropout в eval — тождество);\n    S4     — выход стадии 4 энкодера (вход decode_head): (n, 256) токенов сетки 1/32;\n    Q/K/V  — выходы q_proj / k_proj / v_proj последнего блока стадии 4: (n, 256), 8 голов × 32 склеены;\n    logits — выход decode_head: (19, h/4, w/4) до подъёма разрешения.\n    """\n\n    def __init__(self, name):\n        super().__init__()\n        self.full_res = name in FULL_RES\n        self.net = SegformerForSemanticSegmentation.from_pretrained(CHECKPOINTS[name])\n        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) * 255)\n        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1) * 255)\n        self.features = {}\n        if self.full_res:\n            return\n        att =self.net.segformer.stages[3].blocks[-1].attention\n        for key, mod in (("Q", att.q_proj), ("K", att.k_proj), ("V", att.v_proj)):\n            mod.register_forward_hook(lambda m, i, o, k=key: self.features.__setitem__(k, o[0]))\n        head = self.net.decode_head\n        head.register_forward_pre_hook(lambda m, a: self.features.__setitem__("S4", a[0][-1][0].flatten(1).T))\n        head.classifier.register_forward_pre_hook(lambda m, a: self.features.__setitem__("C", a[0]))\n        head.register_forward_hook(lambda m, i, o: self.features.__setitem__("logits", o[0]))\n\n    def forward(self, image, size):\n        x = (image.float() - self.mean) / self.std\n        logits = self.net(pixel_values=x).logits\n        return F.interpolate(logits, size=(size[1], size[0]), mode="bilinear", align_corners=False)\n\n    @torch.inference_mode()\n    def probs(self, sample, size, full=True):\n        """Элемент датасета → softmax (19, H, W) на GPU на рабочем разрешении size.\n        full=False — тяжёлая модель тоже получает уменьшенный кадр (SegDataset.heavy_full)."""\n        image = sample["image" if self.full_res and full else "small"]\n        return self(image[None].to(self.mean.device), size).softmax(1)[0]\n\n    def tokens(self, key):\n        """Признак как строки (n, 256) float32."""\n        x = self.features[key]\n        if key == "C":\n            x = x[0].flatten(1).T\n        return x.float().contiguous()\n\n    def c_map(self, size, normalize=False):\n        """C, поднятый билинейно до рабочего разрешения: (256, H·W); normalize — L2 по каналам."""\n        up = F.interpolate(self.features["C"].float(), size=(size[1], size[0]), mode="bilinear",\n                           align_corners=False)[0]\n        if normalize:\n            up = F.normalize(up, dim=0)\n        return up.reshape(up.shape[0], -1)\n\n    def neg_lse(self, size):\n        """−logsumexp логитов, поднятых до рабочего разрешения: (H·W,)."""\n        return -upsample_logits(self.features["logits"], size).logsumexp(0).reshape(-1)\n\ndef load_model(name, device="cuda"):\n    return SegModel(name).to(device).eval()\n\ndef entropy_score(probs, dim=0):\n    return -(probs * probs.clamp_min(1e-12).log()).sum(dim)\n\nBLOCKS = [f"s{s}b{j}" for s in range(1, 5) for j in (1, 2)]\n\nSTRIDES = {1: 4, 2: 8, 3: 16, 4: 32}\n\nPATCH = 16\n\ndef stage(block):\n    return int(block[1])\n\ndef stride(block):\n    return STRIDES[stage(block)]\n\ndef fc2(model, block):\n    """fc2 of the block in SegModel model ("s1b1" → stages[0].blocks[0])."""\n    return model.net.segformer.stages[stage(block) - 1].blocks[int(block[3]) - 1].mlp.fc2\n\nclass MixFFNTap:\n    """Context manager over SegModel: during forward passes (batch of 1) self.acts[block] holds the\n    block\'s fc2 input as (H, W, 4C) for the last frame.\n\n    replace = {block: f}: f((N, 4C)) → (N, 4C) is passed to fc2 instead (splice test); acts keeps the\n    original activations.\n    """\n\n    def __init__(self, model, blocks=BLOCKS, replace=None):\n        self.model, self.blocks, self.replace = model, list(blocks), dict(replace or {})\n        self.acts, self.grid, self.handles = {}, {}, []\n\n    def __enter__(self):\n        stages = self.model.net.segformer.stages\n        for s in sorted({stage(b) for b in self.blocks + list(self.replace)}):\n            self.handles.append(stages[s - 1].patch_embeddings.register_forward_hook(\n                lambda m, i, o, s=s: self.grid.__setitem__(s, (o[1], o[2]))))\n        for b in sorted(set(self.blocks) | set(self.replace)):\n            self.handles.append(fc2(self.model, b).register_forward_pre_hook(\n                lambda m, args, b=b: self._pre(b, args)))\n        return self\n\n    def __exit__(self, *exc):\n        for h in self.handles:\n            h.remove()\n        self.handles = []\n\n    def _pre(self, block, args):\n        x = args[0]\n        h, w = self.grid[stage(block)]\n        assert x.shape[0] == 1 and x.shape[1] == h * w, (block, tuple(x.shape), (h, w))\n        if block in self.blocks:\n            self.acts[block] = x[0].reshape(h, w, x.shape[2])\n        if block in self.replace:\n            return (self.replace[block](x[0]).to(x.dtype)[None],) + tuple(args[1:])\n        return None\n\ndef selection_index(n_patches, n_grid, block, patch=PATCH):\n    """Positions of the block\'s grid closest to each patch along one axis: (n_patches, m), m = P/r for\n    r < P (all of them are max-pooled), otherwise m = 1."""\n    r = stride(block)\n    u = np.arange(n_patches)\n    if r < patch:\n        assert patch % r == 0\n        m = patch // r\n        idx = u[:, None] * m + np.arange(m)[None, :]\n    else:\n        idx = np.floor((patch * u + (patch - 1) / 2) / r + 0.5).astype(np.int64)[:, None]\n        idx = np.clip(idx, 0, n_grid - 1)\n    assert idx.max() < n_grid, (block, n_patches, n_grid)\n    return idx\n\ndef pool_to_patches(x, block, n_patches_hw, patch=PATCH, reduce="max"):\n    """x (H, W, D) on the block\'s grid → (Hp·Wp, D): max (or mean) over the closest positions of each patch,\n    raster order of patches."""\n    Hp, Wp = n_patches_hw\n    rows = torch.as_tensor(selection_index(Hp, x.shape[0], block, patch), device=x.device)\n    cols = torch.as_tensor(selection_index(Wp, x.shape[1], block, patch), device=x.device)\n    y = x[rows][:, :, cols]                       # (Hp, m, Wp, m, D)\n    y = y.amax(dim=(1, 3)) if reduce == "max" else y.mean(dim=(1, 3)) if reduce == "mean" else None\n    assert y is not None, reduce\n    return y.reshape(Hp * Wp, x.shape[2])\n\ndef per_patch(x, patch=PATCH):\n    """(H, W) map → (Hp·Wp, P²): pixels of each patch, raster order of patches."""\n    H, W = x.shape\n    assert H % patch == 0 and W % patch == 0, (H, W)\n    return x.reshape(H // patch, patch, W // patch, patch).swapaxes(1, 2).reshape(-1, patch * patch)\n\ndef boundary_map(pred):\n    """Pixels whose 3×3 neighbourhood (inside the frame) has more than one predicted class."""\n    p = np.pad(pred, 1, mode="edge")\n    H, W = pred.shape\n    out = np.zeros((H, W), dtype=bool)\n    for dy in (-1, 0, 1):\n        for dx in (-1, 0, 1):\n            out |= p[1 + dy:1 + dy + H, 1 + dx:1 + dx + W] != pred\n    return out\n\ndef output_features(probs, pred, patch=PATCH):\n    """F0 of every patch from the B0 softmax probs (19, H, W) and pred (H, W) tensors: {name: (n,) numpy}."""\n    ent = per_patch(entropy_score(probs), patch)\n    top2 = probs.topk(2, dim=0).values\n    msp = per_patch(top2[0], patch)\n    margin = per_patch(top2[0] - top2[1], patch)\n    counts = F.one_hot(per_patch(pred, patch), NC).sum(1)\n    boundary = per_patch(torch.from_numpy(boundary_map(pred.cpu().numpy())), patch)\n    Hp, Wp = pred.shape[0] // patch, pred.shape[1] // patch\n    row, col = np.divmod(np.arange(Hp * Wp), Wp)\n    f = {"ent_mean": ent.mean(1), "ent_max": ent.amax(1), "ent_p90": torch.quantile(ent, 0.9, dim=1),\n         "msp_mean": msp.mean(1), "msp_min": msp.amin(1), "margin_mean": margin.mean(1),\n         "margin_min": margin.amin(1), "boundary_frac": boundary.float().mean(1)}\n    f = {k: v.float().cpu().numpy() for k, v in f.items()}\n    f["cls"] = counts.argmax(1).cpu().numpy().astype(np.int16)     # ties → smallest class id\n    f["row"], f["col"] = row.astype(np.int16), col.astype(np.int16)\n    return f\n\nclass _Normalized(nn.Module):\n    def __init__(self, d, mean, scale):\n        super().__init__()\n        self.d = d\n        self.register_buffer("mean", torch.zeros(d) if mean is None else torch.as_tensor(mean).float())\n        self.register_buffer("scale", torch.tensor(float(scale)))\n\n    def normalize(self, h):\n        return (h.float() - self.mean) / self.scale\n\n    def denormalize(self, x):\n        return x * self.scale + self.mean\n\nclass BottleneckAE(_Normalized):\n    kind = "ae"\n\n    def __init__(self, d, b, h=None, mean=None, scale=1.0):\n        super().__init__(d, mean, scale)\n        self.b, self.h = b, h or d\n        self.enc = nn.Sequential(nn.Linear(d, self.h), nn.GELU(), nn.Linear(self.h, b))\n        self.dec = nn.Sequential(nn.Linear(b, self.h), nn.GELU(), nn.Linear(self.h, d))\n\n    def encode(self, x):\n        return self.enc(x)\n\n    def decode(self, z):\n        return self.dec(z)\n\n    def config(self):\n        return {"d": self.d, "b": self.b, "h": self.h}\n\nclass PCA(_Normalized):\n    kind = "pca"\n\n    def __init__(self, d, b, mean=None, scale=1.0):\n        super().__init__(d, mean, scale)\n        self.b = b\n        self.register_buffer("center", torch.zeros(d))          # μ: mean of the normalised training x\n        self.register_buffer("components", torch.zeros(b, d))   # rows: eigenvectors, decreasing eigenvalue\n\n    def encode(self, x):\n        return (x - self.center) @ self.components.T\n\n    def decode(self, z):\n        return z @ self.components + self.center\n\n    def config(self):\n        return {"d": self.d, "b": self.b}\n\nKINDS = {"ae": BottleneckAE, "pca": PCA}\n\ndef load_encoder(path, device="cuda"):\n    ckpt = torch.load(path, map_location=device, weights_only=False)\n    enc = KINDS[ckpt["kind"]](**ckpt["config"]).to(device)\n    enc.load_state_dict(ckpt["state"])\n    return enc.eval(), ckpt\n\ndef scalar_columns(tag):\n    """Names of the R or N columns: <block>_<tag>_mean, <block>_<tag>_max (stages 1–2), <block>_<tag>."""\n    return [c for b in BLOCKS for c in ([f"{b}_{tag}_mean", f"{b}_{tag}_max"] if stride(b) < PATCH else [f"{b}_{tag}"])]\n\ndef pool_scalar(v, block, hw):\n    """Per-position values (h, w) → (n_patches, 2) mean and max (stages 1–2) or (n_patches, 1)."""\n    v = v[..., None]\n    if stride(block) < PATCH:\n        return torch.cat([pool_to_patches(v, block, hw, reduce="mean"), pool_to_patches(v, block, hw, reduce="max")], 1)\n    return pool_to_patches(v, block, hw, reduce="mean")\n\ndef frame_features(acts, encs, norms, hw):\n    """{encoder: (Z, R)} and N of one frame, numpy float32."""\n    Z = {n: [] for n in encs}\n    R = {n: [] for n in encs}\n    N = []\n    for b in BLOCKS:\n        a = acts[b]\n        h, w, d = a.shape\n        x = (a.reshape(-1, d).float() - norms[b][0]) / norms[b][1]\n        N.append(pool_scalar((x.pow(2).sum(1) / d).reshape(h, w), b, hw))\n        for n, e in encs.items():\n            enc = e[b]\n            z = enc.encode(x)\n            r = (enc.decode(z) - x).pow(2).sum(1) / d\n            Z[n].append(pool_to_patches(z.reshape(h, w, -1), b, hw, reduce="mean"))\n            R[n].append(pool_scalar(r.reshape(h, w), b, hw))\n    out = {n: (torch.cat(Z[n], 1).cpu().numpy(), torch.cat(R[n], 1).cpu().numpy()) for n in encs}\n    return out, torch.cat(N, 1).cpu().numpy()\n', 'router.py': '"""Per-frame choice of the compute level under an average budget, online.\n\nLevels (cost = share of one full heavy-model run, on top of the light model):\n  light     the light model alone                                         0\n  ens@0.5   mean of light and heavy probabilities, heavy on a 0.5x frame  0.25\n  ens@0.75  the same with a 0.75x frame                                   0.5625\n  heavy     the heavy model on the full frame                             1\nRule (E10/E11): a frame whose `flag` score is above the clean-data quantile gets ens@0.75 (ens@0.5 or light\nwhen the credit does not allow it); any frame whose `priority` score is in the top share of the recent window\nis upgraded to heavy. The top share follows the budget left after the flagged frames, and a credit (each frame\nadds `budget`, each level spends its cost) keeps the long-run average within the budget.\n"""\nfrom collections import deque\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Level:\n    name: str\n    scale: float = 0.0     # heavy-model input scale, 0 = no heavy run\n    ensemble: bool = False\n\n    @property\n    def cost(self):\n        return self.scale ** 2\n\n\nLEVELS = {l.name: l for l in [Level("light"), Level("ens@0.5", 0.5, True), Level("ens@0.75", 0.75, True),\n                              Level("heavy", 1.0)]}\n\n\nclass Router:\n    def __init__(self, budget=0.5, quantile=0.95, window=50, flagged="ens@0.75", fallback="ens@0.5",\n                 max_debt=1.0, max_credit=2.0):\n        self.budget, self.quantile, self.window = budget, quantile, window\n        self.flagged, self.fallback = LEVELS[flagged], LEVELS[fallback]\n        self.max_debt, self.max_credit = max_debt, max_credit\n        self.threshold = None\n        self.reset()\n\n    def reset(self):\n        """Starts a new stream: empty window and zero credit."""\n        self.credit = 0.0\n        self.priorities, self.flags = deque(maxlen=self.window), deque(maxlen=self.window)\n\n    def fit(self, flag_scores):\n        self.threshold = float(np.quantile(flag_scores, self.quantile))\n\n    def affordable(self, level):\n        return self.credit - level.cost >= -self.max_debt\n\n    def decide(self, flag, priority) -> Level:\n        self.credit = min(self.credit + self.budget, self.max_credit)\n        is_flagged = flag is not None and flag > self.threshold\n        self.flags.append(is_flagged)\n        self.priorities.append(priority)\n\n        base = LEVELS["light"]\n        if is_flagged:\n            base = next(l for l in (self.flagged, self.fallback, base) if self.affordable(l))\n        # share of frames that can go to heavy with the budget left after the flagged ones\n        flagged_cost = np.mean(self.flags) * self.flagged.cost\n        share = np.clip((self.budget - flagged_cost) / (1 - flagged_cost), 0, 1)\n        level = base\n        if share > 0 and priority >= np.quantile(self.priorities, 1 - share) and self.affordable(LEVELS["heavy"]):\n            level = LEVELS["heavy"]\n        self.credit -= level.cost\n        return level\n\n', 'p0_ids.txt': 'frankfurt_000000_000294\nfrankfurt_000000_001016\nfrankfurt_000000_001751\nfrankfurt_000000_002196\nfrankfurt_000000_002963\nfrankfurt_000000_003357\nfrankfurt_000000_003920\nfrankfurt_000000_004617\nfrankfurt_000000_005898\nfrankfurt_000000_006589\nfrankfurt_000000_008206\nfrankfurt_000000_009291\nfrankfurt_000000_009561\nfrankfurt_000000_009688\nfrankfurt_000000_011461\nfrankfurt_000000_011810\nfrankfurt_000000_012009\nfrankfurt_000000_012121\nfrankfurt_000000_013240\nfrankfurt_000000_013382\nfrankfurt_000000_013942\nfrankfurt_000000_015676\nfrankfurt_000000_016005\nfrankfurt_000000_016286\nfrankfurt_000000_017228\nfrankfurt_000000_018797\nfrankfurt_000000_019607\nfrankfurt_000000_020215\nfrankfurt_000000_020321\nfrankfurt_000000_020880\nfrankfurt_000000_021667\nfrankfurt_000000_021879\nfrankfurt_000000_022254\nfrankfurt_000001_002512\nfrankfurt_000001_002646\nfrankfurt_000001_002759\nfrankfurt_000001_003056\nfrankfurt_000001_005184\nfrankfurt_000001_005410\nfrankfurt_000001_005898\nfrankfurt_000001_007285\nfrankfurt_000001_007407\nfrankfurt_000001_007973\nfrankfurt_000001_008688\nfrankfurt_000001_009058\nfrankfurt_000001_009504\nfrankfurt_000001_010156\nfrankfurt_000001_010444\nfrankfurt_000001_010830\nfrankfurt_000001_012699\nfrankfurt_000001_012738\nfrankfurt_000001_012870\nfrankfurt_000001_013710\nfrankfurt_000001_014221\nfrankfurt_000001_014406\nfrankfurt_000001_014565\nfrankfurt_000001_014741\nfrankfurt_000001_015091\nfrankfurt_000001_015328\nfrankfurt_000001_016273\nfrankfurt_000001_016462\nfrankfurt_000001_017101\nfrankfurt_000001_019698\nfrankfurt_000001_020046\nfrankfurt_000001_020287\nfrankfurt_000001_021406\nfrankfurt_000001_023235\nfrankfurt_000001_024927\nfrankfurt_000001_025713\nfrankfurt_000001_025921\nfrankfurt_000001_027325\nfrankfurt_000001_028232\nfrankfurt_000001_028590\nfrankfurt_000001_028854\nfrankfurt_000001_029086\nfrankfurt_000001_029236\nfrankfurt_000001_030067\nfrankfurt_000001_030669\nfrankfurt_000001_031416\nfrankfurt_000001_032556\nfrankfurt_000001_032711\nfrankfurt_000001_034047\nfrankfurt_000001_034816\nfrankfurt_000001_037705\nfrankfurt_000001_038245\nfrankfurt_000001_038418\nfrankfurt_000001_038844\nfrankfurt_000001_039895\nfrankfurt_000001_040732\nfrankfurt_000001_041074\nfrankfurt_000001_041354\nfrankfurt_000001_041517\nfrankfurt_000001_042384\nfrankfurt_000001_043395\nfrankfurt_000001_044227\nfrankfurt_000001_044413\nfrankfurt_000001_044525\nfrankfurt_000001_044658\nfrankfurt_000001_044787\nfrankfurt_000001_046126\nfrankfurt_000001_046504\nfrankfurt_000001_047178\nfrankfurt_000001_047552\nfrankfurt_000001_048196\nfrankfurt_000001_049298\nfrankfurt_000001_051516\nfrankfurt_000001_052120\nfrankfurt_000001_052594\nfrankfurt_000001_054219\nfrankfurt_000001_054415\nfrankfurt_000001_055062\nfrankfurt_000001_055172\nfrankfurt_000001_055709\nfrankfurt_000001_056580\nfrankfurt_000001_057478\nfrankfurt_000001_057954\nfrankfurt_000001_058057\nfrankfurt_000001_058176\nfrankfurt_000001_058504\nfrankfurt_000001_058914\nfrankfurt_000001_059119\nfrankfurt_000001_059642\nfrankfurt_000001_059789\nfrankfurt_000001_060422\nfrankfurt_000001_060545\nfrankfurt_000001_060906\nfrankfurt_000001_061763\nfrankfurt_000001_062016\nfrankfurt_000001_062250\nfrankfurt_000001_062509\nfrankfurt_000001_063045\nfrankfurt_000001_064130\nfrankfurt_000001_064305\nfrankfurt_000001_064651\nfrankfurt_000001_064798\nfrankfurt_000001_065160\nfrankfurt_000001_065850\nfrankfurt_000001_066438\nfrankfurt_000001_066574\nfrankfurt_000001_067092\nfrankfurt_000001_067295\nfrankfurt_000001_067474\nfrankfurt_000001_067735\nfrankfurt_000001_070099\nfrankfurt_000001_071781\nfrankfurt_000001_072295\nfrankfurt_000001_073243\nfrankfurt_000001_073911\nfrankfurt_000001_075296\nfrankfurt_000001_075984\nfrankfurt_000001_076502\nfrankfurt_000001_077092\nfrankfurt_000001_077233\nfrankfurt_000001_077434\nfrankfurt_000001_080091\nfrankfurt_000001_080830\nfrankfurt_000001_082087\nfrankfurt_000001_083029\nfrankfurt_000001_083852\nlindau_000002_000019\nlindau_000005_000019\nlindau_000006_000019\nlindau_000007_000019\nlindau_000010_000019\nlindau_000011_000019\nlindau_000014_000019\nlindau_000017_000019\nlindau_000018_000019\nlindau_000019_000019\nlindau_000023_000019\nlindau_000024_000019\nlindau_000025_000019\nlindau_000026_000019\nlindau_000028_000019\nlindau_000029_000019\nlindau_000030_000019\nlindau_000031_000019\nlindau_000032_000019\nlindau_000033_000019\nlindau_000036_000019\nlindau_000038_000019\nlindau_000039_000019\nlindau_000041_000019\nlindau_000043_000019\nlindau_000046_000019\nlindau_000049_000019\nlindau_000053_000019\nlindau_000054_000019\nlindau_000057_000019\nlindau_000058_000019\nmunster_000001_000019\nmunster_000002_000019\nmunster_000003_000019\nmunster_000004_000019\nmunster_000005_000019\nmunster_000008_000019\nmunster_000009_000019\nmunster_000010_000019\nmunster_000011_000019\nmunster_000012_000019\nmunster_000013_000019\nmunster_000014_000019\nmunster_000016_000019\nmunster_000017_000019\nmunster_000019_000019\nmunster_000020_000019\nmunster_000021_000019\nmunster_000022_000019\nmunster_000023_000019\nmunster_000027_000019\nmunster_000028_000019\nmunster_000029_000019\nmunster_000030_000019\nmunster_000032_000019\nmunster_000033_000019\nmunster_000035_000019\nmunster_000036_000019\nmunster_000039_000019\nmunster_000040_000019\nmunster_000041_000019\nmunster_000042_000019\nmunster_000045_000019\nmunster_000046_000019\nmunster_000047_000019\nmunster_000048_000019\nmunster_000049_000019\nmunster_000052_000019\nmunster_000053_000019\nmunster_000054_000019\nmunster_000055_000019\nmunster_000056_000019\nmunster_000057_000019\nmunster_000059_000019\nmunster_000060_000019\nmunster_000061_000019\nmunster_000062_000019\nmunster_000065_000019\nmunster_000067_000019\nmunster_000068_000019\nmunster_000070_000019\nmunster_000071_000019\nmunster_000073_000019\nmunster_000076_000019\nmunster_000077_000019\nmunster_000078_000019\nmunster_000079_000019\nmunster_000080_000019\nmunster_000081_000019\nmunster_000082_000019\nmunster_000083_000019\nmunster_000084_000019\nmunster_000085_000019\nmunster_000087_000019\nmunster_000089_000019\nmunster_000092_000019\nmunster_000093_000019\nmunster_000094_000019\nmunster_000100_000019\nmunster_000103_000019\nmunster_000106_000019\nmunster_000108_000019\nmunster_000109_000019\nmunster_000110_000019\nmunster_000111_000019\nmunster_000113_000019\nmunster_000114_000019\nmunster_000115_000019\nmunster_000119_000019\nmunster_000122_000019\nmunster_000123_000019\nmunster_000124_000019\nmunster_000127_000019\nmunster_000128_000019\nmunster_000129_000019\nmunster_000130_000019\nmunster_000131_000019\nmunster_000133_000019\nmunster_000134_000019\nmunster_000139_000019\nmunster_000142_000019\nmunster_000143_000019\nmunster_000144_000019\nmunster_000145_000019\nmunster_000148_000019\nmunster_000149_000019\nmunster_000150_000019\nmunster_000151_000019\nmunster_000152_000019\nmunster_000154_000019\nmunster_000157_000019\nmunster_000159_000019\nmunster_000160_000019\nmunster_000161_000019\nmunster_000162_000019\nmunster_000164_000019\nmunster_000165_000019\nmunster_000166_000019\nmunster_000167_000019\nmunster_000171_000019\nmunster_000172_000019\n\n', 'features/__init__.py': '', 'features/base.py': 'from torch import nn\n\n\nclass Feature(nn.Module):\n    """forward(out) scores one light-model output (models.common.Out, batch 1) as a 0-dim tensor,\n    higher = the heavy model is more needed. `taps` names the light-model taps the feature reads.\n    A feature with a fit accumulates clean frames in update(out) and completes the fit in finish();\n    fitted tensors are buffers, so they are saved with state_dict()."""\n    taps = ()\n\n    def update(self, out):\n        pass\n\n    def finish(self):\n        pass\n\n', 'features/entropy.py': 'from features.base import Feature\n\n\nclass Entropy(Feature):\n    """Mean per-pixel entropy of the light model\'s softmax."""\n\n    def forward(self, out):\n        logp = out.logits[0].log_softmax(0)\n        return -(logp.exp() * logp).sum(0).mean()\n\n', 'features/knn.py': 'import torch\nimport torch.nn.functional as F\n\nfrom features.base import Feature\n\n\nclass KNN(Feature):\n    """Mean cosine distance of the decoder features (1/8 resolution) to their K nearest neighbours in a bank\n    of features sampled from clean frames: high when the frame does not look like the fit data."""\n    taps = ("decoder",)\n\n    def __init__(self, bank_size=20000, per_frame=500, k=10, seed=0):\n        super().__init__()\n        self.bank_size, self.per_frame, self.k = bank_size, per_frame, k\n        self.gen = torch.Generator().manual_seed(seed)\n        self.samples = []\n        self.register_buffer("bank", torch.empty(0))  # (bank_size, channels), fp16, unit norm; size set by the fit\n        self.register_load_state_dict_pre_hook(\n            lambda m, state, prefix, *_: setattr(m, "bank", torch.empty_like(state[prefix + "bank"], device=m.bank.device)))\n\n    @staticmethod\n    def vectors(out):\n        f = F.avg_pool2d(out.taps["decoder"][0].float(), 2)  # 1/4 -> 1/8\n        return F.normalize(f.flatten(1).T, dim=1)\n\n    def update(self, out):\n        v = self.vectors(out).cpu()\n        self.samples.append(v[torch.randperm(len(v), generator=self.gen)[:self.per_frame]])\n\n    def finish(self):\n        v = torch.cat(self.samples)\n        self.bank = v[torch.randperm(len(v), generator=self.gen)[:self.bank_size]].to(self.bank.device, torch.half)\n        self.samples = []\n\n    def forward(self, out):\n        sim = (self.vectors(out).to(self.bank) @ self.bank.T).topk(self.k, dim=1).values.float()\n        return (1 - sim.mean(1)).mean()\n\n', 'guide.md': '# Integration guide: B0 patch error score (R, Z, ZR, F3F0NR)\n\nWritten in ASD-STE100 (Simplified Technical English). Status: 2026-10-28, branch `SAE`, commit `1df870a`.\n\n## 1. Scope\n\nThis guide tells you how to install the patch error scorer in a router. The guide does not know the router.\nThus, the scorer has one input (a frame) and one main output (a score map). You connect the score map to\nyour router units with an adapter (section 8).\n\nThe scorer finds 16 × 16 pixel patches where SegFormer-B0 makes errors. It uses four feature sets:\n\n| Set | Features | Columns |\n|---|---|---|\n| R | B0 output statistics (F0), activation norm (N), AE reconstruction residual (R) | 11 + 12 + 12 = 35 |\n| Z | F0, AE bottleneck codes (Z) | 11 + 512 = 523 |\n| ZR | F0, N, Z, R | 11 + 12 + 512 + 12 = 547 |\n| F3F0NR | Raw activations (raw), F0, N, R | 4,096 + 11 + 12 + 12 = 4,131 |\n\nA CatBoost model gives the score from these features. The score is the predicted fraction of wrong B0\npixels in the patch.\n\n## 2. Data that the scorer gets\n\nThe scorer gets these data for each frame:\n\n1. **The input frame.** RGB, uint8, shape (3, H, W), at the working resolution of B0.\n   - Cityscapes: 1024 × 512 (the 2048 × 1024 frame, bilinear resize).\n   - ACDC: 1024 × 576 (the 1920 × 1080 frame, bilinear resize).\n2. **The B0 softmax.** Shape (19, H, W). B0 is `nvidia/segformer-b0-finetuned-cityscapes-512-1024`, fp32.\n3. **The Mix-FFN hidden activations of B0.** This is the input of `fc2` in each of the 8 encoder blocks\n   (s1b1 … s4b2). A forward pre-hook catches them during the usual B0 forward pass. The hook does not change\n   the B0 output (the logits difference is 0).\n\n| Stage | Blocks | Stride | Channels d | Code size b = d / 8 |\n|---|---|---|---|---|\n| 1 | s1b1, s1b2 | 4 | 128 | 16 |\n| 2 | s2b1, s2b2 | 8 | 256 | 32 |\n| 3 | s3b1, s3b2 | 16 | 640 | 80 |\n| 4 | s4b1, s4b2 | 32 | 1024 | 128 |\n\nThe scorer does not use ground truth. It does not use a heavy model.\n\n## 3. How the scorer processes the data\n\nThe scorer does these steps for each frame:\n\n1. It runs B0 one time with the hooks. It keeps the softmax and the 8 activation maps.\n2. It normalises each activation x = (h − mean) / scale. The constants are in the encoder checkpoints.\n3. It encodes and decodes x with the bottleneck autoencoder of the block. The AE is\n   d → d (GELU) → b → d (GELU) → d.\n4. For each native position, it calculates:\n   - Z: the code z (b values).\n   - R: the residual ‖x − x̂‖² / d.\n   - N: the norm ‖x‖² / d.\n   - raw (F3F0NR only): the activation h itself, not normalised (d values).\n5. It moves the values to the 16 × 16 patch grid:\n   - Stages 1–2: N and R use the mean and the max over the positions in the patch. Z uses the mean. raw uses\n     the max.\n   - Stages 3–4: the scorer uses the one position nearest to the patch centre.\n   - raw is then rounded to fp16, as in training.\n6. It calculates F0 from the B0 softmax for each patch:\n   - entropy: mean, max, p90;\n   - max softmax: mean, min;\n   - top-2 margin: mean, min;\n   - boundary fraction;\n   - row and column of the patch;\n   - majority class `cls` (a categorical column).\n7. It puts the columns in the model order: F0, then N, then Z, then R. For F3F0NR the order is raw, F0, N, R.\n8. It applies the CatBoost model and returns the probability.\n\n## 4. Output\n\n`ErrorScorer(small)` returns a dict:\n\n| Key | Shape | Type | Content |\n|---|---|---|---|\n| `score` | (Hp, Wp) | float32 | Predicted fraction of wrong B0 pixels in the patch, 0…1. Higher = more errors. |\n| `pred` | (H, W) | int64, GPU | B0 prediction (argmax). This is the usual B0 output. |\n| `probs` | (19, H, W) | float32, GPU | B0 softmax. |\n| `N` | (Hp·Wp, 12) | float32 | Activation norms (section 3, step 4). |\n| `Z` | (Hp·Wp, 512) | float32 | AE codes. |\n| `R` | (Hp·Wp, 12) | float32 | AE residuals. |\n| `raw` | (Hp·Wp, 4096) | float32 | Raw activations. Only for F3F0NR. |\n\nHp = H / 16 and Wp = W / 16. Patches are in raster order: patch = row · Wp + col. For Cityscapes, the grid\nis 32 × 64. For ACDC, it is 36 × 64.\n\nUse `score` if your router needs one number for each region. Use `N`, `Z`, `R` (and `raw`) if your router\ntrains its own model.\n\nThe column names are:\n- N: `<block>_n_mean`, `<block>_n_max` (stages 1–2), `<block>_n` (stages 3–4);\n- R: the same with `r`;\n- Z: `<block>_z<j>`;\n- raw: `<block>_h<j>`.\n\nAll columns are in the block order s1b1 … s4b2.\n\n## 5. Files to copy\n\n### 5.1 Code\n\nCopy these objects from branch `SAE` into one module, `ae_error/lib.py`. Copy them without changes. Copy\nonly the objects in the list. The full files import training code that the scorer does not need.\n\n| From | Objects |\n|---|---|\n| `core/components.py` | `NC` (= 19) |\n| `core/models.py` | `CHECKPOINTS`, `FULL_RES`, `upsample_logits`, `SegModel`, `load_model` |\n| `core/evaluation.py` | `entropy_score` |\n| `sae/hooks.py` | `BLOCKS`, `STRIDES`, `PATCH`, `stage`, `stride`, `fc2`, `MixFFNTap`, `selection_index`, `pool_to_patches` |\n| `sae/patch_features.py` | `per_patch`, `boundary_map`, `output_features` |\n| `ae/model.py` | `_Normalized`, `BottleneckAE`, `PCA`, `KINDS`, `load_encoder` |\n| `ae/patch_features.py` | `scalar_columns`, `pool_scalar`, `frame_features` |\n\nPut these imports at the top of `ae_error/lib.py`:\n\n```python\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom transformers import SegformerForSemanticSegmentation\n```\n\nNOTE: The router can already have `core/models.py` and `core/evaluation.py`. If so, import `SegModel`,\n`load_model` and `entropy_score` from there. Make sure that `SegModel` is the same as on branch `SAE`.\n\nPut an empty `ae_error/__init__.py` next to it. Then add the scorer, `ae_error/scorer.py`. This is new code.\nCopy it from this guide:\n\n```python\n"""Patch error score of SegFormer-B0 from bottleneck-AE features (R, Z, ZR, F3F0NR)."""\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom catboost import CatBoost, Pool\n\nfrom ae_error.lib import (BLOCKS, PATCH, MixFFNTap, frame_features, load_encoder, load_model, output_features,\n                          pool_to_patches)\n\nF0 = ["ent_mean", "ent_max", "ent_p90", "msp_mean", "msp_min", "margin_mean", "margin_min",\n      "boundary_frac", "row", "col", "cls"]\nPARTS = {"R": ("F0", "N", "R"), "Z": ("F0", "Z"), "ZR": ("F0", "N", "Z", "R"), "F3F0NR": ("raw", "F0", "N", "R")}\n\n\nclass ErrorScorer:\n    """features: "R", "Z", "ZR" or "F3F0NR"; root: the results/ae folder of the SAE project, with\n    ckpt/ae_b8/<block>.pt and runs/<features>_ae_b8_catboost/model.cbm."""\n\n    def __init__(self, features, root="results/ae", device="cuda", b0=None):\n        root = Path(root)\n        self.parts = PARTS[features]\n        self.b0 = b0 if b0 is not None else load_model("b0", device)\n        self.enc = {b: load_encoder(root / "ckpt" / "ae_b8" / f"{b}.pt", device)[0] for b in BLOCKS}\n        self.norms = {b: (e.mean, e.scale.item()) for b, e in self.enc.items()}\n        self.model = CatBoost()\n        self.model.load_model(str(root / "runs" / f"{features}_ae_b8_catboost" / "model.cbm"))\n\n    @torch.inference_mode()\n    def __call__(self, small):\n        """small: (3, H, W) uint8 RGB frame at the working resolution, H and W multiples of 16.\n        Returns score (Hp, Wp), B0 pred (H, W), probs (19, H, W) and the per-patch features N, Z, R\n        (and raw for F3F0NR)."""\n        H, W = small.shape[1:]\n        hw = (H // PATCH, W // PATCH)\n        with MixFFNTap(self.b0) as tap:\n            probs = self.b0(small[None].to(self.b0.mean.device), (W, H)).softmax(1)[0]\n            pred = probs.argmax(0)\n            enc_out, N = frame_features(tap.acts, {"ae": self.enc}, self.norms, hw)\n            if "raw" in self.parts:      # max over the patch positions, stored as fp16 (as in training)\n                raw = np.concatenate([pool_to_patches(tap.acts[b].float(), b, hw).half().cpu().numpy()\n                                      for b in BLOCKS], axis=1).astype(np.float32)\n        Z, R = enc_out["ae"]\n        f0 = output_features(probs, pred)\n        feats = {"N": N, "Z": Z, "R": R, **({"raw": raw} if "raw" in self.parts else {})}\n        cols = {"F0": pd.DataFrame({c: f0[c] for c in F0}), **{k: pd.DataFrame(v) for k, v in feats.items()}}\n        X = pd.concat([cols[p] for p in self.parts], axis=1)\n        X.columns = self.model.feature_names_\n        X["cls"] = X["cls"].astype(np.int32)\n        score = self.model.predict(Pool(X, cat_features=["cls"]), prediction_type="Probability")[:, 1]\n        return {"score": score.reshape(hw).astype(np.float32), "pred": pred, "probs": probs, **feats}\n```\n\n### 5.2 Fitted artifacts\n\nCAUTION: The weights are not in git (`results/` is in `.gitignore`). You must download them yourself. Get the\narchive `ae_error_artifacts.zip` from the file exchange (ask the owner of branch `SAE` for the link).\n\nThe archive keeps the paths of this project. It contains:\n\n| Path in the archive | Content | Size |\n|---|---|---|\n| `results/ae/ckpt/ae_b8/s1b1.pt` … `s4b2.pt`, 8 files | Bottleneck AE of each block, with the normalisation constants | 27 MB |\n| `results/ae/runs/R_ae_b8_catboost/model.cbm` | CatBoost model, set R | 0.6 MB |\n| `results/ae/runs/Z_ae_b8_catboost/model.cbm` | CatBoost model, set Z | 1.5 MB |\n| `results/ae/runs/ZR_ae_b8_catboost/model.cbm` | CatBoost model, set ZR | 1.4 MB |\n| `results/ae/runs/F3F0NR_ae_b8_catboost/model.cbm` | CatBoost model, set F3F0NR | 1.3 MB |\n| `results/ae/runs/<set>_ae_b8_catboost/scores.parquet`, 4 files | Saved test scores, only for the acceptance test (section 9) | 4 × 20 MB |\n\nWhere to unpack:\n- **Clone of this project:** unpack the archive into the repository root (the folder with `pyproject.toml`).\n  The files then are at the same paths as on the experiment machine, and the scorer default\n  `root="results/ae"` finds them when you start Python from the repository root.\n- **Router repository:** unpack the archive into the router repository root, or into a different folder. Give\n  the `results/ae` folder to the scorer: `ErrorScorer("ZR", root="<folder>/results/ae")`.\n\nDo not change the folder structure under `results/ae`. The scorer finds the files by these paths. It needs\nonly `ckpt/ae_b8/` and the `model.cbm` of your set (32 MB in total). The `scores.parquet` files are only for\nthe acceptance test.\n\n### 5.3 Python packages\n\n`torch`, `transformers`, `catboost`, `numpy`, `pandas`, `pillow`, and `pyarrow` for the acceptance test\n(it reads `.parquet` files). The tested versions are torch 2.14.1+cu130, transformers 5.20.2, catboost 1.2.10,\npandas 3.0.6, pyarrow 26.0.0 and Python 3.14. The torch wheel comes from the index\n`https://download.pytorch.org/whl/cu130` (see `pyproject.toml` on branch `SAE`).\n\nNOTE: At the first start, `load_model("b0")` downloads the B0 weights from Hugging Face. Make sure that the\nmachine has network access or a Hugging Face cache that contains them.\n\n## 6. Procedure\n\n1. Copy the code (section 5.1) into the router repository.\n2. Download the archive and unpack it (section 5.2).\n3. Install the Python packages (section 5.3).\n4. Make one scorer for each process:\n\n   ```python\n   from ae_error.scorer import ErrorScorer\n   scorer = ErrorScorer("ZR", root="results/ae")      # or "R", "Z", "F3F0NR"\n   ```\n\n5. If the router already has a B0 `SegModel`, give it to the scorer. This prevents a second copy of B0:\n\n   ```python\n   scorer = ErrorScorer("ZR", root="results/ae", b0=router_b0)\n   ```\n\n6. Replace the router\'s B0 call with the scorer call:\n\n   ```python\n   out = scorer(small)              # small: (3, H, W) uint8, working resolution\n   pred, score = out["pred"], out["score"]\n   ```\n\n7. Use `out["pred"]` and `out["probs"]` as the B0 result. Do not run B0 a second time.\n8. Do the acceptance test (section 9).\n\n## 7. Cautions and limits\n\nCAUTION: Give the scorer one frame at a time. `MixFFNTap` accepts only a batch of 1. It stops with an\nassertion if the batch is larger.\n\nCAUTION: Use the working resolution of section 2. The models know only 1024 × 512 (Cityscapes) and\n1024 × 576 (ACDC). The `row` and `col` features are patch positions. Thus, a different resolution changes\ntheir meaning.\n\nCAUTION: H and W must be multiples of 16. If not, `per_patch` stops with an assertion.\n\nCAUTION: Use transformers 5.x. `MixFFNTap` uses the module names of transformers 5.x\n(`segformer.stages[s].patch_embeddings`, `blocks[j].mlp.fc2`) and the output (embeddings, height, width) of\nthe patch embeddings. Another version can stop with an error or give wrong hook positions. After an upgrade,\ndo the acceptance test (section 9) again.\n\nCAUTION: Use B0 in fp32. We did not test fp16 or other B0 checkpoints. If you change B0, you must train the\nencoders and CatBoost again (section 11).\n\n- The score predicts the **B0 error** in the patch. It does not predict the gain from a heavy model. If the\n  router needs the gain, train a small model on top of the score or the features.\n- The models are fitted on Cityscapes only: AE on 2,975 train frames, CatBoost on 300 val frames (P0). Do\n  not use the Cityscapes P0 frames to test the router. The list is in `splits/cityscapes_p0.txt`.\n- The scorer gives a score to all patches, also to patches without ground truth.\n- We tested the scorer only on CUDA.\n\n## 8. Adapters to router units\n\nThe router can use a different unit than the 16 × 16 patch. Use these adapters (tested):\n\n```python\nimport numpy as np\n\ndef to_pixels(score, patch=16):\n    """(Hp, Wp) patch scores -> (H, W) pixel map."""\n    return np.repeat(np.repeat(score, patch, axis=0), patch, axis=1)\n\ndef region_stats(pixel_map, regions, n):\n    """regions: (H, W) int unit ids 0..n-1 (components, tiles, ...). Mean, max and expected wrong pixels."""\n    r, v = regions.ravel(), pixel_map.ravel()\n    area = np.bincount(r, minlength=n)\n    total = np.bincount(r, weights=v, minlength=n)\n    peak = np.full(n, -np.inf)\n    np.maximum.at(peak, r, v)\n    return {"mean": total / np.maximum(area, 1), "max": peak, "expected_errors": total}\n```\n\n| Router unit | What to do |\n|---|---|\n| 16 × 16 patch | Use `score` directly. |\n| Square tile (multiple of 16, for example 256) | Make a tile id map. Apply `region_stats`. A 256 tile holds 16 × 16 patches. |\n| Connected component or other region | Apply `region_stats` with the component map. |\n| Frame | Use `score.mean()` or the sum of `expected_errors`. |\n\n`expected_errors` is the predicted number of wrong B0 pixels in the unit. Use it if the router sorts units\nunder a pixel budget. Use `mean` if the router sorts units of equal size.\n\nTo give the features to a learned router, aggregate the N, Z, R (and raw) columns the same way. Reshape each column\nto (Hp, Wp). Then apply `to_pixels` and `region_stats`.\n\n## 9. Acceptance test\n\nDo this test after the installation. It compares the scorer with the saved test scores of branch `SAE`.\n\n1. Use the saved scores `results/ae/runs/<set>_ae_b8_catboost/scores.parquet` from the archive (section 5.2).\n2. Load a frame and make it the same way as the B0 pipeline:\n\n   ```python\n   img = Image.open(path).convert("RGB").resize((1024, 512), Image.BILINEAR)   # ACDC: (1024, 576)\n   small = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)\n   ```\n\n3. Get `out = scorer(small)`.\n4. Get the rows of `scores.parquet` for this `image_id`. For ACDC, `image_id` is `<condition>_<frame>`.\n5. Compare `out["score"].ravel()[rows.patch]` with `rows.score`.\n6. Make sure that the maximum difference is less than 1e-6.\n\nOur result (2026-10-28, outside the repository, with only the copied code): the maximum difference was\n≤ 3e-8 for R, Z, ZR and F3F0NR. We tested Cityscapes frame `frankfurt_000000_000576` and ACDC frame\n`night_GOPR0351_frame_000033`. The difference comes from the float32 output.\n\n## 10. Which set to use, and what it costs\n\nQuality: the scorers are fitted on Cityscapes P0 and tested on P1 and ACDC val (RESULTSV2.md, RESULTSv2.1.md).\n- Recall@10% is the share of wrong pixels in the top 10% of patches.\n- Robust AUROC is the y50 AUROC inside entropy deciles 4–9.\n\n| Set | P1 recall@10% | ACDC recall@10% | P1 robust AUROC | ACDC robust AUROC |\n|---|---|---|---|---|\n| entropy only | 0.552 | 0.300 | 0.571 | 0.561 |\n| R | 0.564 | 0.313 | 0.743 | 0.737 |\n| Z | 0.563 | 0.304 | 0.731 | 0.642 |\n| ZR | 0.566 | 0.319 | 0.744 | 0.687 |\n| F3F0NR | 0.562 | 0.299 | 0.733 | 0.590 |\n\nACDC recall@10% per condition, where the sets differ most:\n\n| Set | fog | night | rain | snow |\n|---|---|---|---|---|\n| entropy only | 0.420 | 0.171 | 0.422 | 0.354 |\n| R | 0.428 | 0.168 | 0.418 | 0.342 |\n| Z | 0.424 | 0.168 | 0.442 | 0.360 |\n| ZR | 0.428 | 0.177 | 0.435 | 0.356 |\n| F3F0NR | 0.426 | 0.166 | 0.446 | 0.365 |\n\nTime per frame: RTX 3060 Ti, fp32, batch 1, 1024 × 512, mean of 20 Cityscapes frames, the copied package:\n\n| Part | R | Z | ZR | F3F0NR |\n|---|---|---|---|---|\n| B0 only | 17 ms | 17 ms | 17 ms | 17 ms |\n| B0 + hooks + features | 46 ms | 46 ms | 46 ms | 80 ms |\n| Full scorer (+ CatBoost) | 55 ms | 67 ms | 68 ms | 253 ms |\n\nF3F0NR is slow. The raw pooling adds approximately 35 ms. The table of 4,131 columns and the CatBoost step add\napproximately 170 ms.\n\nRecommendation:\n- **ZR**: the best recall on P1 and ACDC. Use it as the default.\n- **R**: the best ranking inside the same entropy level under domain shift. It is also the fastest.\n- **Z**: no advantage over ZR at the same cost.\n- **F3F0NR**: the best recall in rain and snow. On ACDC all, it is worse than R and ZR (RESULTSV2.md §6a), and it\n  is about 4 times slower. Use it only if your data are mostly rain or snow, and if you can accept the time.\n\nNOTE: F3F0NR is a descriptive set. We added it after the test results were known.\n\n## 11. Train again (only if B0, the resolution or the data change)\n\nRun these commands on branch `SAE`. The training writes the files to the paths of section 5.2. Then make a new\narchive from these paths, relative to the repository root. Do not put `results/ae/ckpt/ae_b8/train.csv` or\n`results/ae/acts/` in the archive. The scorer does not use them.\n\nBefore you start, put the data into `data/` (it is in `.gitignore`):\n- Cityscapes: `leftImg8bit/train`, `leftImg8bit/val` and `gtFine/val`.\n- ACDC: `rgb_anon` and `gt` of fog, night, rain and snow, `val`.\n\n`download.cityscapes` and `download.acdc` get these files. By default, `download.cityscapes` gets only\n`leftImg8bit/val/`. Use `--prefix leftImg8bit/train/` for the train frames. Use `--package` and `--prefix`\n(`--pattern` for ACDC) for the label archives.\n\n```bash\nuv run python -m ae.collect\nuv run python -m ae.train\nuv run python -m sae.patch_features --dataset cityscapes --sae none\nuv run python -m sae.patch_features --dataset acdc --sae none\nuv run python -m ae.patch_features --dataset cityscapes --encoders ae_b8\nuv run python -m ae.patch_features --dataset acdc --encoders ae_b8\nuv run python -m ae.classify cv --features ZR --encoder ae_b8 --model catboost\nuv run python -m ae.classify test --features ZR --encoder ae_b8 --model catboost\n```\n\nNOTE: `ae.classify` reads F0 and the labels from the v1 base tables. `sae.patch_features --sae none` writes\nonly these tables (it does not need the SAE). Run it before `ae.patch_features`. `ae.classify test` scores\nCityscapes P1 and all ACDC conditions. Thus, it needs the ACDC tables too. Repeat the last two commands for R,\nZ and F3F0NR. F3F0NR also reads the raw activations from the v1 base tables.\n\n'}
for name, source in SOURCES.items():
    path = EXP / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert path.read_text() == source, f'Другая версия {path}; не смешивай эксперименты'
    else:
        path.write_text(source)
os.environ.update({
    'ZR_ROOT': str(ROOT), 'ZR_OUT': str(OUT), 'ZR_ART': str(ART),
    'ZR_CS': str(CS), 'ZR_ACDC': str(ACDC),
})


## 2. Чекпоинты AE и готовые CatBoost Z/R/ZR

Папка пользователя: https://drive.google.com/drive/folders/1O2z2d5xcK6Fzsxu6JgariOqLZXr4bw7I

Восемь AE используют один и тот же набор нормализаций из чекпоинтов. `scores.parquet` нужны только для acceptance; они не используются для выбора регионов/кадров и не заменяют инференс. Если файлы есть от старого ноутбука или в архиве результатов, переиспользуем. Для Z метаданные файла берём из подпапки runs, а не угадываем идентификатор.

In [ ]:
DOWNLOAD_SOURCE = "from pathlib import Path\nimport os, shutil, zipfile\nimport gdown\nroot=Path(os.environ['ZR_ROOT']);art=Path(os.environ['ZR_ART'])\ndef download(file_id,path):\n    if path.is_file():return\n    path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_suffix(path.suffix+'.part')\n    result=gdown.download(id=file_id,output=str(temporary),quiet=False,resume=True)\n    assert result and temporary.is_file(), f'Не удалось скачать {path.name}'\n    temporary.replace(path)\nae_ids={\n's1b1':'19cMwkPSqt3m0X1BuKGp3fBajiFi1j1vi','s1b2':'12Euk64imwprlW8B6pgJMG249w5QW0gXu',\n's2b1':'1EjOLVaA6LLl8ckVRI2QUahVypuFP5GDn','s2b2':'1gX4u9JKfrQqqAscr-nSHPl1fbsmL1LOF',\n's3b1':'1EGA0Mx4-v_509IIA0B53PHaSZKdWAuL4','s3b2':'1P-plpLzDi4RoO5tgozdtsnY-A-qmgxQt',\n's4b1':'116bvp72j6_OusC6TuHk5II0e-3-ToZQv','s4b2':'1HQR0wmkffK0DXtL5YkdtEP3RUNS59SKl'}\nfor block,file_id in ae_ids.items():\n    dst=art/'ckpt/ae_b8'/(block+'.pt');dst.parent.mkdir(parents=True,exist_ok=True)\n    if not dst.exists():\n        for old in [Path('/content/spatial_allblock_gain_v1/ae_b8'),Path('/content/spatial_ae_acdc_v1/ae_b8')]:\n            src=old/dst.name\n            if src.is_file():shutil.copyfile(src,dst);break\n    download(file_id,dst)\nknown={'R':{'model.cbm':'1M10UwLtVTS3LGSvrWd0jj7DjjOS86Wte','scores.parquet':'1lxcg8tjbp20l3wgxx_HFv1dx5P12WdO3'},\n       'ZR':{'model.cbm':'1bvkTnda0S_9_4U7QOsno8e9fQq0e3dGM','scores.parquet':'1NFS5wFLrP9h4o1s9uJlGjnP35ertnyk8'}}\nfor kind,files in known.items():\n    for name,file_id in files.items():\n        dst=art/'runs'/(kind+'_ae_b8_catboost')/name;dst.parent.mkdir(parents=True,exist_ok=True)\n        old=Path('/content/spatial_ae_acdc_v1/ready_scores')/kind/name\n        if not dst.exists() and old.is_file():shutil.copyfile(old,dst)\n        download(file_id,dst)\nz_files=['model.cbm','scores.parquet'];meta_path=root/'z_drive_files.json'\nif not all((art/'runs/Z_ae_b8_catboost'/name).is_file() for name in z_files):\n    if meta_path.exists():entries=__import__('json').loads(meta_path.read_text())\n    else:\n        listing=gdown.download_folder(url='https://drive.google.com/drive/folders/1oIWJ_w83HNOmfnlRVLyd96wd4DWwHSaw',output=str(root/'drive_listing'),skip_download=True,quiet=True,timeout=90)\n        assert listing is not None,'Не удалось прочитать список готовых моделей'\n        entries=[{'id':f.id,'path':f.path} for f in listing]\n        meta_path.write_text(__import__('json').dumps(entries,indent=2))\n    for name in z_files:\n        matches=[e for e in entries if Path(e['path']).name==name and Path(e['path']).parent.name in ['Z','Z_ae_b8_catboost']]\n        assert len(matches)==1, 'Неоднозначный путь Z: '+str([e['path'] for e in entries])\n        download(matches[0]['id'],art/'runs/Z_ae_b8_catboost'/name)\nprint('8 AE и Z/R/ZR загружены. Переобучение не выполнялось.',flush=True)\n"
p = EXP/'download_artifacts.py'
p.write_text(DOWNLOAD_SOURCE)
subprocess.run([str(PYTHON), '-u', str(p)], env=os.environ.copy(), check=True)


## 3. Данные

Можно указать уже распакованные датасеты. Для Cityscapes нужен полный val500 (скрипт сам исключит пересечение P0), для ACDC только RGB val и gt labelTrainIds val406. Датасеты не нужно грузить на Drive заново. Автоматически проверяем несколько прежних локальных путей; при отсутствии скачиваем архивы на локальный диск и извлекаем только val.

Если публичный gdown не пропускает RGB15.6ГиБ, укажи доступный архив с Drive в `ACDC_ARCHIVES`. Не копируй15ГиБ в заполненный Drive, используй существующий файл/ярлык в смонтированном аккаунте.


In [ ]:
CS_OVERRIDE = ''       # корень с leftImg8bit/val и gtFine/val
ACDC_OVERRIDE = ''     # корень с rgb_anon и gt
CS_ARCHIVES = '/content/drive/MyDrive/datasets/cityscapes'
ACDC_ARCHIVES = ''     # например /content/drive/MyDrive/datasets/acdc
os.environ.update({'ZR_CS_OVERRIDE': CS_OVERRIDE, 'ZR_ACDC_OVERRIDE': ACDC_OVERRIDE,
                   'ZR_CS_ARCHIVES': CS_ARCHIVES, 'ZR_ACDC_ARCHIVES': ACDC_ARCHIVES})
DATA_SOURCE = "from pathlib import Path\nimport os,shutil,zipfile\nimport gdown\ndef download(file_id,path):\n    if path.is_file():return\n    path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_suffix(path.suffix+'.part')\n    result=gdown.download(id=file_id,output=str(temporary),quiet=False,resume=True)\n    assert result and temporary.is_file(), f'Не удалось скачать {path.name}'\n    temporary.replace(path)\n\nroot=Path(os.environ['ZR_ROOT'])\ncs=Path(os.environ['ZR_CS']);acdc=Path(os.environ['ZR_ACDC'])\ncs_override=os.environ.get('ZR_CS_OVERRIDE','');acdc_override=os.environ.get('ZR_ACDC_OVERRIDE','')\ncs_candidates=([Path(cs_override)] if cs_override else [])+[cs,Path('/content/spatial_allblock_gain_v1/data/cityscapes'),Path('/content/spatial_ae_acdc_v1/data/cityscapes')]\nacdc_candidates=([Path(acdc_override)] if acdc_override else [])+[acdc,Path('/content/spatial_allblock_gain_v1/data/acdc'),Path('/content/spatial_ae_acdc_v1/data/acdc'),Path('/content/acdc_e12_local/data/acdc')]\nfor p in cs_candidates:\n    if len(list((p/'leftImg8bit/val').rglob('*_leftImg8bit.png')))==500:\n        cs=p;break\nfor p in acdc_candidates:\n    if len(list((p/'rgb_anon').rglob('*_rgb_anon.png')))>=406 and len(list((p/'gt').rglob('*_gt_labelTrainIds.png')))>=406:\n        acdc=p;break\narchives=root/'archives';archives.mkdir(exist_ok=True)\ndef obtain(name,file_id,extra):\n    dst=archives/name\n    if not dst.is_file():\n        roots=[Path(x) for x in extra if x]+[Path('/content/spatial_allblock_gain_v1/archives'),Path('/content/spatial_ae_acdc_v1/archives'),Path('/content/acdc_archives')]\n        for p in roots:\n            if (p/name).is_file():\n                assert shutil.disk_usage(root).free>(p/name).stat().st_size+2*2**30,'Не хватает локального места'\n                shutil.copyfile(p/name,dst);break\n        if not dst.is_file():\n            assert shutil.disk_usage(root).free>25*2**30,'Оставь 25ГиБ локально для большого RGB архива'\n            download(file_id,dst)\n    assert zipfile.is_zipfile(dst),f'Invalid ZIP {dst}'\n    return dst\ndef unpack(archive,base,prefix,suffix):\n    count=0\n    with zipfile.ZipFile(archive) as z:\n        chosen=[]\n        for member in z.infolist():\n            parts=Path(member.filename).parts\n            if member.is_dir():continue\n            if prefix=='leftImg8bit' and parts and parts[0]=='val':rel=Path('leftImg8bit',*parts)\n            elif prefix in parts:rel=Path(*parts[parts.index(prefix):])\n            else:continue\n            wanted=rel.parts[1]=='val' if prefix in ['leftImg8bit','gtFine'] else len(rel.parts)>3 and rel.parts[1] in ['fog','night','rain','snow'] and rel.parts[2]=='val'\n            if wanted and rel.name.endswith(suffix):\n                assert not rel.is_absolute() and '..' not in rel.parts\n                chosen.append((member,base/rel))\n        need=sum(member.file_size for member,p in chosen if not p.is_file())\n        assert shutil.disk_usage(root).free>need+2**30,'Нет места для извлечения val'\n        for member,dst in chosen:\n            if not dst.is_file():\n                dst.parent.mkdir(parents=True,exist_ok=True)\n                temp=dst.with_suffix('.png.part')\n                with z.open(member) as source,temp.open('wb') as target:shutil.copyfileobj(source,target)\n                temp.replace(dst)\n            count+=1\n    print(prefix,'val files',count,flush=True)\nif len(list((cs/'leftImg8bit/val').rglob('*_leftImg8bit.png')))!=500:\n    archive=obtain('val.zip','1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ',[os.environ.get('ZR_CS_ARCHIVES','')]);unpack(archive,cs,'leftImg8bit','_leftImg8bit.png')\nif len(list((cs/'gtFine/val').rglob('*_gtFine_labelIds.png')))!=500:\n    archive=obtain('gtFine_trainvaltest.zip','1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk',[os.environ.get('ZR_CS_ARCHIVES','')]);unpack(archive,cs,'gtFine','_gtFine_labelIds.png')\nif len(list((acdc/'gt').rglob('*_gt_labelTrainIds.png')))<406:\n    archive=obtain('gt_trainval.zip','1_SqW2pcVXq3cvUPlf0a9KuTQO5SlmirG',[os.environ.get('ZR_ACDC_ARCHIVES','')]);unpack(archive,acdc,'gt','_gt_labelTrainIds.png')\nif len(list((acdc/'rgb_anon').rglob('*_rgb_anon.png')))<406:\n    archive=obtain('rgb_anon_trainvaltest.zip','1RauTG86LkGR1o64RWfDRG1mz_2loclVx',[os.environ.get('ZR_ACDC_ARCHIVES','')]);unpack(archive,acdc,'rgb_anon','_rgb_anon.png')\n(root/'dataset_paths.json').write_text(__import__('json').dumps({'cs':str(cs),'acdc':str(acdc)},indent=2))\nprint('Data ready:',cs,acdc,flush=True)\n"
p=EXP/'prepare_data.py'
p.write_text(DATA_SOURCE)
subprocess.run([str(PYTHON), '-u', str(p)], env=os.environ.copy(), check=True)
paths=json.loads((ROOT/'dataset_paths.json').read_text())
os.environ.update({'ZR_CS': paths['cs'], 'ZR_ACDC': paths['acdc']})


In [ ]:
def run(stage):
    path=OUT/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск:',stage,'| лог:',path,flush=True)
    started=time.monotonic()
    with path.open('w') as log:
        process=subprocess.Popen([str(PYTHON),'-u',str(EXP/'experiment.py'),stage],
            cwd=EXP,env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30);break
                except subprocess.TimeoutExpired:
                    tail=path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{stage}: {(time.monotonic()-started)/60:.1f} мин | '+' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    print(path.read_text(errors='replace')[-15000:])
    assert status==0,f'Ошибка: {path}. Следующий этап не запускать.'
    print('Готово за',round((time.monotonic()-started)/60,1),'мин')
run('preflight')


## 4. Обязательная проверка Z/R/ZR

Проверяем совпадение с эталонным `scores.parquet` на6кадрах: Frankfurt000576, первый каждого ACDC условия и night_GOPR0351_frame_000033 автора, если он не первый. Проверяем все3модели, не только ZR, и неизменность B0 от хуков.

**Каждый max_error должен быть <1e-6.** При ошибке присылай `acceptance.json` и лог: следующие этапы заблокированы. Версии/HF GPU всё равно нужно проверить на реальных данных; здесь нет обещания битового совпадения на другой видеокарте. Порог не ослабляем, веса не переобучаем.

In [ ]:
run('acceptance')
print((OUT/'acceptance.json').read_text())

## 5. Подгонка банка и порогов на Frankfurt267

БанкkNN — точный алгоритм main:20тыс. нормированных векторов декодера, k=10, seed0. Для Z/R/ZR используем **готовые** CatBoost и одинаковые правила усреднения карты; q95 каждого признака на чистых кадрах. Ни GT, ни ACDC метрики для подгонки не читаются. Это не обучение нейросетей. При повторном запуске подгонка будет выполнена заново с фиксированным seed; не меняй её после оценки ACDC.

In [ ]:
run('fit')
print((OUT/'thresholds.json').read_text())

## 6. Итоговое качество

Один проход B0 с hooks/AE даёт карты всех3scorer; готовые CatBoost R/Z применяются к тем же N/Z/R и F0, без повторных B0. Сохраняем для каждого кадра матрицы ошибок всех4уровней B5 и все оценки. Это позволяет воспроизвести5роутеров и6порядков потока без новых прогонов.

Время этого общего кэша — **не латентность отдельного метода**. Каждый кадр включает до3B5проходов; общий этап может занять десятки минут. Прогресс даёт фактическую скорость. Завершённые кадры пропускаются при повторном запуске. Локальный кэш нужен для повторного анализа/замеров; сброс среды его удалит.

После кэша автоматически считаются mIoU и bootstrap. На этом этапе CPU-анализ может некоторое время не печатать прогресс; heartbeat показывает, что процесс работает. Сохранённый общий mIoU читается из `final_miou.csv`, интервалы — `bootstrap.csv`.

In [ ]:
run('cache')
# pandas в kernel нужен только для отображения; можно вместо этого открыть CSV в Drive.
import pandas as pd
summary=pd.read_csv(OUT/'final_miou.csv')
display(summary[(summary.order=='blocked') & summary.domain.isin(['acdc_all','pool'])])
bs=pd.read_csv(OUT/'bootstrap.csv')
display(bs[(bs.order=='blocked') & (bs.domain=='pool') & (bs.unit=='sequence')])


## 6а. Доля закрытого разрыва B0→B5, CPU

`gap_closed_percent = 100 × (mIoU_pipeline − mIoU_B0)/(mIoU_B5 − mIoU_B0)`.

Считаем отдельно для каждого порядка, условия, ACDC и общего пула из тех же кадров. B0=0%, B5=100%; отрицательные значения и значения выше100% не обрезаем. При почти нулевом знаменателе метрика не определена. Если B5 не лучше B0, это явно помечается и долю нельзя трактовать как восстановление преимущества тяжёлой модели.

Интервалы в прежнем `bootstrap.csv` остаются интервалами **разницы mIoU**. Они не превращаются в интервалы доли разрыва простым делением границ.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

summary = pd.read_csv(OUT/'final_miou.csv')
references = summary.pivot(index=['order','domain'], columns='method', values='miou')
references = references[['b0','b5_full']].rename(columns={'b0':'miou_b0','b5_full':'miou_b5'}).reset_index()
with_gap = summary.merge(references, on=['order','domain'], validate='many_to_one')
denominator = with_gap.miou_b5 - with_gap.miou_b0
with_gap['gain_vs_b0_pp'] = with_gap.miou - with_gap.miou_b0
with_gap['gap_closed_percent'] = np.divide(
    100*with_gap.gain_vs_b0_pp.to_numpy(), denominator.to_numpy(),
    out=np.full(len(with_gap), np.nan), where=np.abs(denominator.to_numpy())>1e-6,
)
with_gap['heavy_better_than_b0'] = denominator > 1e-6
with_gap.to_csv(OUT/'final_miou_with_gap.csv', index=False)
display(with_gap[(with_gap.order=='blocked') & with_gap.domain.isin(['acdc_all','pool'])])


## 7. Сквозное время каждого варианта, GPU

40детерминированных кадров:8из каждого набора,2повтора, прогрев. Для каждого метода восстанавливаем его состояние непосредственно перед кадром в блочном потоке (вне часов), затем отдельно измеряем: CPU RGB decode/resize → один B0 → нужный scorer/kNN/entropy → решение Router → выбранный B5/ансамбль → маскаCPU. GT/метрики вне часов. Порядок методов перемешан.

Сравниваем живое решение уровня и матрицу ошибок с кэшем. При расхождении не продолжаем. Не используем ускоренный совместный кэш как время Z/R/ZR.

Среднее/p95 относятся к фиксированной подвыборке40, а качество — ко всем498кадрам. Расходы по площади публикуются отдельно и могут различаться при одинаковом номинальном budget0.5.

In [ ]:
run('timing')
display(pd.read_csv(OUT/'timing.csv'))

## 7а. Кривые бюджета: качество из кэша, CPU

У Ottobiss **нет числа окон**. Меняем только номинальный бюджет B5:0.1/0.25/0.5/0.75/1.0. Порог Z/R/ZR/kNN и веса остаются прежними. B0 и полную B5 добавляем отдельными контрольными точками. Нулевой бюджет не имитируем через Router с долгом: точка0 — действительно B0 без роутера.

Повторных B0/B5 для качества не нужно. Состояние роутера пересчитывается на том же блочном потоке для каждого бюджета. Все бюджеты исследовательские; не выбираем победителя для заявления на новой выборке по этому графику. Скрипт живёт отдельно и не меняет протокол предыдущих стадий.

In [ ]:
SWEEP_SOURCE = '"""Read cached levels; sweep unchanged router budgets; time live paths separately."""\nimport os,sys,argparse,json,copy,time\nfrom pathlib import Path\nsys.path.insert(0,os.environ[\'ZR_EXPERIMENTS\'])\nimport experiment as m\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom router import Router,LEVELS\n\nBUDGETS=[.1,.25,.5,.75,1.]\nOUT=m.OUT/\'budget_sweep_v1\'\ndef prepare():\n OUT.mkdir(parents=True,exist_ok=True)\n manifest=json.loads((m.OUT/\'manifest.json\').read_text())[\'test\']\n state=torch.load(m.OUT/\'state.pt\',map_location=\'cpu\',weights_only=True)\n assert state[\'protocol_sha\']==m.sha(m.OUT/\'protocol.json\')\n m.require_acceptance()\n selected=[]\n for d in m.DOMAINS:\n  ix=[i for i,r in enumerate(manifest) if r[\'domain\']==d]\n  selected += [ix[j] for j in np.linspace(0,len(ix)-1,min(8,len(ix))).astype(int)]\n m.fixed(OUT/\'protocol.json\',dict(source_sha=m.sha(__file__),parent_protocol_sha=m.sha(m.OUT/\'protocol.json\'),state_sha=m.sha(m.OUT/\'state.pt\'),budgets=BUDGETS,methods=m.METHODS,order=\'blocked\',timing_names=[manifest[i][\'name\'] for i in selected],repeats=2,quality=\'all498 common frames; thresholds fixed on cs_fit\',timing=\'fixed40 frames, blocked stream history; CPU decode to CPU mask; not population mean\',status=\'exploratory budget sweep; do not select/refit on evaluation\'))\n scores=[];confs=[]\n for r in manifest:\n  with np.load(m.CACHE/(r[\'name\']+\'.npz\'),allow_pickle=False) as z:scores.append(z[\'scores\']);confs.append(z[\'confs\'])\n return manifest,state,selected,np.array(scores),np.array(confs)\ndef routers(state,budget):\n result={}\n for method in m.METHODS:\n  r=Router(budget,m.QUANTILE,m.WINDOW)\n  if method!=\'entropy\':r.threshold=state[\'thresholds\'][method]\n  result[method]=r\n return result\n\ndef gap(value,light,heavy):\n den=heavy-light\n return float(100*(value-light)/den) if abs(den)>1e-6 else np.nan\n\ndef quality():\n manifest,state,selected,scores,all_conf=prepare();domains=np.array([r[\'domain\'] for r in manifest]);names=np.array([r[\'name\'] for r in manifest]);summary=[]\n for budget in BUDGETS:\n  rs=routers(state,budget);cs=np.zeros((len(manifest),len(m.METHODS),19,19),np.int64);levels=np.zeros((len(manifest),len(m.METHODS)),int);cost=levels.astype(float)\n  for i,r in enumerate(manifest):\n   for j,method in enumerate(m.METHODS):\n    level=rs[method].decide(m.flag(scores[i],method),float(scores[i,0]));index=m.MODES.index(level.name);levels[i,j]=index;cost[i,j]=level.cost;cs[i,j]=all_conf[i,index]\n  m.save(OUT/f\'budget_{budget:g}_confusions.npz\',confs=cs,cost=cost,levels=levels,names=names,methods=np.array(m.METHODS))\n  for domain in m.DOMAINS+[\'acdc_all\',\'pool\']:\n   ix=np.ones(len(manifest),bool) if domain==\'pool\' else domains!=\'cs_test\' if domain==\'acdc_all\' else domains==domain\n   light=m.miou(all_conf[ix,0].sum(0));heavy=m.miou(all_conf[ix,-1].sum(0))\n   for j,method in enumerate(m.METHODS):\n    value=m.miou(cs[ix,j].sum(0));summary.append(dict(domain=domain,method=method,budget=budget,miou=value,miou_b0=light,miou_b5=heavy,gain_vs_b0_pp=value-light,gap_closed_percent=gap(value,light,heavy),mean_b5_area_cost=float(cost[ix,j].mean()),frames=int(ix.sum())))\n  print(\'Quality budget\',budget,\'done\',flush=True)\n for domain in m.DOMAINS+[\'acdc_all\',\'pool\']:\n  ix=np.ones(len(manifest),bool) if domain==\'pool\' else domains!=\'cs_test\' if domain==\'acdc_all\' else domains==domain\n  light=m.miou(all_conf[ix,0].sum(0));heavy=m.miou(all_conf[ix,-1].sum(0))\n  for method,value,budget in [(\'b0\',light,0.),(\'b5_full\',heavy,1.)]:summary.append(dict(domain=domain,method=method,budget=budget,miou=value,miou_b0=light,miou_b5=heavy,gain_vs_b0_pp=value-light,gap_closed_percent=gap(value,light,heavy),mean_b5_area_cost=budget,frames=int(ix.sum())))\n pd.DataFrame(summary).to_csv(OUT/\'quality.csv\',index=False)\n # Preserve reference confusions for later gap/bootstrap without the big local cache.\n m.save(OUT/\'reference_confusions.npz\',light=all_conf[:,0],heavy=all_conf[:,-1],names=names,groups=np.array([r[\'sequence\'] for r in manifest]))\n print(pd.DataFrame(summary).query("domain==\'pool\'").to_string(index=False))\n\n@torch.inference_mode()\ndef timing():\n assert torch.cuda.is_available(),\'GPU required for real timing\'\n m.setup();manifest,state,selected,scores,all_conf=prepare();engine=m.Engine();m.load_state(engine)\n keys=[(b,method) for b in BUDGETS for method in m.METHODS]+[(0.,\'b0\'),(1.,\'b5_full\')]\n snapshots={};decisions={}\n for b in BUDGETS:\n  rs=routers(state,b)\n  for i in range(len(manifest)):\n   for method in m.METHODS:\n    if i in selected:snapshots[b,i,method]=copy.deepcopy(rs[method])\n    level=rs[method].decide(m.flag(scores[i],method),float(scores[i,0]))\n    if i in selected:decisions[b,i,method]=m.MODES.index(level.name)\n def call(r,method,router):\n  torch.cuda.synchronize();start=time.perf_counter();img=m.small(r)\n  if method==\'b5_full\':answer=engine.heavy_probs(img,1.).argmax(0).byte().cpu().numpy();index=3\n  elif method==\'b0\':answer=engine.b0(img[None].cuda(),tuple(r[\'size\'])).softmax(1).argmax(1)[0].byte().cpu().numpy();index=0\n  else:\n   kind=method.split(\'_\')[0] if method.split(\'_\')[0] in m.KINDS else None\n   probs,out,ent=engine.forward(img,kind)\n   fl=float(out[\'score\'].mean()) if kind else float(engine.knn(engine.native())) if method==\'knn_entropy\' else None\n   decision=router.decide(fl,ent);answer=engine.predict(img,probs,decision);index=m.MODES.index(decision.name)\n  torch.cuda.synchronize();return answer,index,(time.perf_counter()-start)*1000\n for b,method in keys:call(manifest[selected[0]],method,copy.deepcopy(snapshots.get((b,selected[0],method))))\n for pos,i in enumerate(selected):\n  dest=OUT/\'timing_records\'/(manifest[i][\'name\']+\'.json\')\n  rec=json.loads(dest.read_text()) if dest.exists() else {}\n  tasks=[(b,method,rep) for b,method in keys for rep in range(2)];np.random.default_rng(100000+i).shuffle(tasks)\n  gt=None\n  for b,method,rep in tasks:\n   key=f\'{b:g}|{method}|{rep}\'\n   if key in rec:continue\n   answer,index,ms=call(manifest[i],method,copy.deepcopy(snapshots.get((b,i,method))))\n   expected=0 if method==\'b0\' else 3 if method==\'b5_full\' else decisions[b,i,method]\n   assert index==expected,f\'Live routing mismatch: {manifest[i]["name"]} {b} {method}\'\n   if gt is None:gt=m.ground(manifest[i])\n   assert np.array_equal(m.conf(gt,answer),all_conf[i,index]),f\'Live prediction mismatch: {manifest[i]["name"]} {method}\'\n   rec[key]=float(ms);m.write(dest,rec)\n  print(\'Timing sweep\',pos+1,\'/\',len(selected),flush=True)\n summary=[]\n for b,method in keys:\n  times=[]\n  for i in selected:\n   rec=json.loads((OUT/\'timing_records\'/(manifest[i][\'name\']+\'.json\')).read_text());times += [rec[f\'{b:g}|{method}|{rep}\'] for rep in range(2)]\n  summary.append(dict(method=method,budget=b,mean_ms=float(np.mean(times)),p95_ms=float(np.percentile(times,95)),timing_frames=len(selected),repeats=2))\n pd.DataFrame(summary).to_csv(OUT/\'timing.csv\',index=False)\n q=pd.read_csv(OUT/\'quality.csv\');combined=q.merge(pd.DataFrame(summary),on=[\'method\',\'budget\'],validate=\'many_to_one\');combined.to_csv(OUT/\'curve_metrics.csv\',index=False);print(combined.query("domain==\'pool\'").to_string(index=False))\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'quality\',\'timing\']);a=p.parse_args();torch.set_num_threads(4);globals()[a.stage]()\n'
sweep_script = ROOT/'budget_sweep/budget_sweep.py'
sweep_script.parent.mkdir(parents=True, exist_ok=True)
if sweep_script.exists():
    assert sweep_script.read_text()==SWEEP_SOURCE, 'Не смешивай версии sweep'
else:
    sweep_script.write_text(SWEEP_SOURCE)
os.environ['ZR_EXPERIMENTS'] = str(EXP)

def run_sweep(stage):
    path=OUT/f'sweep_{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск sweep:',stage,'| лог:',path,flush=True)
    started=time.monotonic()
    with path.open('w') as log:
        process=subprocess.Popen([str(PYTHON),'-u',str(sweep_script),stage],
            cwd=str(EXP),env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30);break
                except subprocess.TimeoutExpired:
                    print(f'{(time.monotonic()-started)/60:.1f} мин | '+' | '.join(path.read_text(errors='replace').splitlines()[-2:]),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    print(path.read_text(errors='replace')[-14000:])
    assert status==0,f'Ошибка: {path}; следующий этап не запускать'

run_sweep('quality')


## 7б. Полное время каждой точки кривой, GPU

Это дополнительные реальные замеры:5методов×5бюджетов и2контроля,40фиксированных кадров×2повтора. Они могут занять существенно больше времени, чем один замер budget0.5. Завершённые отдельные измерения сохраняются и пропускаются при повторном запуске.

Не оцениваем задержку умножением времени полной B5 на долю площади. Для каждой точки реально запускаем нужный путь; проверяем уровень и итоговую маску относительно кэша. История потока восстанавливается перед измеряемым кадром. Среднее/p95 относятся к40кадрам, mIoU — ко всем498.

In [ ]:
run_sweep('timing')

## 7в. Графики и таблицы, CPU

Получаем PNG и PDF:

1. mIoU — среднее полное время;
2. доля закрытого разрыва — среднее полное время;
3. mIoU — фактическая средняя стоимость B5 по площади.

Цифры рядом с точками — номинальные бюджеты, не число окон. Отдельные B0/B5 — контрольные точки. По времени рисуем только блочный поток, для которого есть живые замеры. p95 сохраняется в CSV, но не используется как доверительный интервал среднего.

Не сравнивай эти точки напрямую со старыми опубликованными числами main: здесь согласованный fp32 протокол и тест без P0.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

curve_dir = OUT/'budget_sweep_v1'
curves = pd.read_csv(curve_dir/'curve_metrics.csv')
figure_dir = curve_dir/'figures'
figure_dir.mkdir(exist_ok=True)
colors = {'entropy':'#64748b','knn_entropy':'#2563eb','Z_entropy':'#ea580c','R_entropy':'#059669','ZR_entropy':'#9333ea'}
labels = {'entropy':'Entropy','knn_entropy':'kNN + entropy','Z_entropy':'Z + entropy','R_entropy':'R + entropy','ZR_entropy':'ZR + entropy'}

for domain in ['pool','acdc_all']:
    data = curves[curves.domain==domain].copy()
    # Timing is the common 40-frame subset, not an ACDC-only sample.
    scope = 'mIoU: 498 frames' if domain=='pool' else 'mIoU: 406 ACDC frames'
    for name, xcol, ycol, xlabel, ylabel in [
        ('miou_time','mean_ms','miou','Mean full time, ms','mIoU, %'),
        ('gap_time','mean_ms','gap_closed_percent','Mean full time, ms','B0-to-B5 gap closed, %'),
        ('miou_cost','mean_b5_area_cost','miou','Mean B5 input-area cost','mIoU, %'),
    ]:
        fig, ax = plt.subplots(figsize=(9,6))
        for method, color in colors.items():
            rows = data[data.method==method].sort_values('budget')
            ax.plot(rows[xcol],rows[ycol],'-o',color=color,label=labels[method],markersize=5)
            for _, point in rows.iterrows():
                if point.budget in [.25,.5,1.]:
                    ax.annotate(f'{point.budget:g}',(point[xcol],point[ycol]),xytext=(4,5),textcoords='offset points',fontsize=7,color=color)
        for method, marker, color, label in [('b0','s','#111827','B0 alone'),('b5_full','*','#dc2626','Full B5 alone')]:
            rows = data[data.method==method]
            ax.scatter(rows[xcol],rows[ycol],marker=marker,s=110,color=color,label=label,zorder=5)
        timing_note = 'time: common 40-frame mixed subset, 2 repeats' if xcol=='mean_ms' else 'B5 area proxy; B0/features excluded from this cost'
        ax.set(title=f'{domain}: blocked stream\n{scope}; {timing_note}',xlabel=xlabel,ylabel=ylabel)
        ax.grid(alpha=.25);ax.legend(fontsize=8,ncol=2)
        fig.tight_layout()
        for extension in ['png','pdf']:
            fig.savefig(figure_dir/f'{domain}_{name}.{extension}',dpi=180,bbox_inches='tight')
        plt.show()
        plt.close(fig)

display(curves[curves.domain=='pool'][['method','budget','miou','gap_closed_percent','mean_b5_area_cost','mean_ms','p95_ms']])
print('Графики:',figure_dir)


## 7г. Дополнительный анализ по децилям энтропии

Для каждого16×16патча берём среднюю энтропию полной рабочей карты B0. Пороги10групп оцениваем на267чистых Cityscapes без GT. Одни и те же группы используются у entropy, kNN, Z, R, ZR и B0/B5. На ACDC это не обязательно10равных по размеру групп: сдвиг может сосредоточить патчи в высоких децилях; ничьи также могут оставлять группы пустыми.

Для каждого дециля складываем матрицы ошибок всех его пикселей и считаем mIoU. Затем считаем равновесное среднее по непустым децилям. Показываем число патчей, валидных пикселей, ошибок B0, кадров и GT-классов. Маленькие группы не используем для уверенных выводов. Это условная метрика с другим весом областей, **не стандартный глобальный mIoU**.

Роутер и его решения не меняются: оцениваем фиксированный блочный пайплайн budget0.5 внутри каждого диапазона уверенности. Это не эксперимент с отдельным выбором патчей внутри дециля. Карты Z/R/ZR сохраняются для последующего разбора, но по тестовым группам ничего не подбираем.

Понадобится дополнительный GPU-проход: из одной покадровой матрицы ошибок нельзя восстановить пространственное расположение ошибок. Сначала B0 на clean267для порогов, затем B0/scorer и необходимые уровни B5 на498кадрах. Совпадение сумм децильных матриц с исходными матрицами проверяется точно. Время этого диагностического прохода не выдаём за латентность метода. Завершённые кадры сохраняются и пропускаются при повторном запуске. Этап необязательный и не блокирует основные финальные метрики.

In [ ]:
DECILE_SOURCE = '"""Conditional mIoU of fixed baseline/Z/R/ZR pipelines on clean entropy bins."""\nimport os,sys,argparse,json,time\nfrom pathlib import Path\nsys.path.insert(0,os.environ[\'ZR_EXPERIMENTS\'])\nimport experiment as m\nimport numpy as np\nimport pandas as pd\nimport torch\n\nOUT=m.OUT/\'entropy_deciles_v1\';CACHE=m.ROOT/\'entropy_deciles_cache_v1\'\nMETHODS=m.METHODS+[\'b0\',\'b5_full\']\ndef patch_mean(value):\n h,w=value.shape;assert h%16==0 and w%16==0\n return value.reshape(h//16,16,w//16,16).mean((1,3))\ndef binned_conf(gt,pred,bins):\n valid=gt!=255;idx=bins[valid].astype(np.int64)*361+gt[valid].astype(np.int64)*19+pred[valid].astype(np.int64)\n return np.bincount(idx,minlength=10*361).reshape(10,19,19)\ndef prepare():\n OUT.mkdir(parents=True,exist_ok=True);CACHE.mkdir(parents=True,exist_ok=True)\n m.require_acceptance();manifest=json.loads((m.OUT/\'manifest.json\').read_text());state=torch.load(m.OUT/\'state.pt\',map_location=\'cpu\',weights_only=True)\n assert state[\'protocol_sha\']==m.sha(m.OUT/\'protocol.json\')\n m.fixed(OUT/\'protocol.json\',dict(source_sha=m.sha(__file__),parent_protocol_sha=m.sha(m.OUT/\'protocol.json\'),state_sha=m.sha(m.OUT/\'state.pt\'),methods=METHODS,patch=16,bins=\'mean B0 full-working-size entropy per16patch; q10..q90 on cs_fit267, no GT\',test=\'same498 frames; same blocked budget0.5 pipeline levels\',macro=\'equal-weight mean of nonempty decile mIoUs, report bin counts; not global mIoU\',status=\'exploratory conditional diagnostic; no tuning on test; no latency claim\'))\n return manifest[\'fit\'],manifest[\'test\']\n\n@torch.inference_mode()\ndef thresholds():\n fit,test=prepare();m.setup();engine=m.Engine();values=[]\n for i,r in enumerate(fit):\n  path=CACHE/\'clean\'/(r[\'name\']+\'.npy\');path.parent.mkdir(parents=True,exist_ok=True)\n  if path.exists():values.append(np.load(path,allow_pickle=False));continue\n  probs,_,_=engine.forward(m.small(r));ent=-(probs*probs.clamp_min(1e-12).log()).sum(0).cpu().numpy();a=patch_mean(ent).astype(np.float32);tmp=path.with_suffix(\'.npy.part\')\n  with tmp.open(\'wb\') as f:np.save(f,a)\n  tmp.replace(path);values.append(a)\n  if (i+1)%20==0:print(\'Clean entropy\',i+1,\'/\',len(fit),flush=True)\n cuts=np.quantile(np.concatenate([a.ravel() for a in values]),np.arange(1,10)/10)\n m.write(OUT/\'thresholds.json\',dict(cuts=cuts.tolist(),fit_frames=len(fit),ties=\'searchsorted side=right; repeated cuts can produce empty bins\'))\n print(\'Entropy cuts:\',cuts,flush=True)\n\n@torch.inference_mode()\ndef evaluate():\n fit,test=prepare();m.setup();engine=m.Engine();m.load_state(engine);cuts=np.array(json.loads((OUT/\'thresholds.json\').read_text())[\'cuts\'])\n with np.load(m.OUT/\'blocked_confusions.npz\',allow_pickle=False) as z:\n  assert list(z[\'names\'])==[r[\'name\'] for r in test];levels=z[\'levels\'];assert list(z[\'methods\'])==m.METHODS\n start=time.monotonic()\n for i,r in enumerate(test):\n  dest=CACHE/\'test\'/(r[\'name\']+\'.npz\')\n  if dest.exists():continue\n  img=m.small(r);probs,maps,ent,knn=engine.all_scores(img)\n  actual=np.array([ent,knn]+[float(maps[k].mean()) for k in m.KINDS])\n  with np.load(m.CACHE/(r[\'name\']+\'.npz\'),allow_pickle=False) as z:\n   assert np.allclose(actual,z[\'scores\'],rtol=0,atol=1e-6),f\'Scores changed: {r["name"]}\'\n   expected=z[\'confs\'].copy()\n  entropy=-(probs*probs.clamp_min(1e-12).log()).sum(0).cpu().numpy();pe=patch_mean(entropy);pb=np.searchsorted(cuts,pe,side=\'right\');bins=np.repeat(np.repeat(pb,16,0),16,1);gt=m.ground(r)\n  predictions={0:probs.argmax(0).byte().cpu().numpy()}\n  for index in sorted(set(levels[i].tolist()+[3])):\n   if index not in predictions:predictions[index]=engine.predict(img,probs,m.LEVELS[m.MODES[index]])\n  method_indices=levels[i].tolist()+[0,3];confs=[]\n  for method,index in zip(METHODS,method_indices):\n   c=binned_conf(gt,predictions[index],bins);assert np.array_equal(c.sum(0),expected[index]),f\'Conditional/global mismatch: {r["name"]} {method}\';confs.append(c)\n  valid=(gt!=255).reshape(gt.shape[0]//16,16,gt.shape[1]//16,16).sum((1,3))>0\n  m.save(dest,confs=np.stack(confs),patch_count=np.bincount(pb.ravel(),minlength=10),valid_patch_count=np.bincount(pb[valid],minlength=10),patch_scores=np.stack([pe]+[maps[k] for k in m.KINDS]))\n  if (i+1)%5==0 or i+1==len(test):print(\'Decile evaluation\',i+1,\'/\',len(test),\'elapsed\',round((time.monotonic()-start)/60,1),\'min\',flush=True)\n analyze()\n\ndef analyze():\n fit,test=prepare();cs=[];patches=[];validpatches=[]\n for r in test:\n  with np.load(CACHE/\'test\'/(r[\'name\']+\'.npz\'),allow_pickle=False) as z:cs.append(z[\'confs\']);patches.append(z[\'patch_count\']);validpatches.append(z[\'valid_patch_count\'])\n cs=np.array(cs);patches=np.array(patches);validpatches=np.array(validpatches);names=np.array([r[\'name\'] for r in test]);groups=np.array([r[\'sequence\'] for r in test]);domains=np.array([r[\'domain\'] for r in test]);cuts=np.array(json.loads((OUT/\'thresholds.json\').read_text())[\'cuts\'])\n m.save(OUT/\'decile_confusions.npz\',confs=cs,names=names,groups=groups,domains=domains,methods=np.array(METHODS),cuts=cuts,patch_count=patches,valid_patch_count=validpatches)\n rows=[];macro=[]\n for domain in m.DOMAINS+[\'acdc_all\',\'pool\']:\n  ix=np.ones(len(test),bool) if domain==\'pool\' else domains!=\'cs_test\' if domain==\'acdc_all\' else domains==domain\n  total=cs[ix].sum(0);valid=total[-2].sum((1,2));error=valid-np.trace(total[-2],axis1=1,axis2=2);nonempty=valid>0\n  for j,method in enumerate(METHODS):\n   values=[]\n   for d in range(10):\n    value=m.miou(total[j,d]) if nonempty[d] else np.nan;values.append(value)\n    rows.append(dict(domain=domain,method=method,decile=d+1,miou=value,patches=int(patches[ix,d].sum()),valid_patches=int(validpatches[ix,d].sum()),valid_pixels=int(valid[d]),b0_error_pixels=int(error[d]),gt_classes=int((total[-2,d].sum(1)>0).sum()),frames_with_valid_pixels=int((cs[ix,-2,d].sum((1,2))>0).sum()),scope=\'fixed clean entropy bins; conditional, not a new primary metric\'))\n   macro.append(dict(domain=domain,method=method,mean_decile_miou=float(np.mean(np.array(values)[nonempty])),nonempty_deciles=int(nonempty.sum()),global_miou=m.miou(total[j].sum(0)),frames=int(ix.sum())))\n pd.DataFrame(rows).to_csv(OUT/\'decile_miou.csv\',index=False);pd.DataFrame(macro).to_csv(OUT/\'macro_decile_miou.csv\',index=False);print(pd.DataFrame(macro).query("domain in [\'acdc_all\',\'pool\']").to_string(index=False))\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'thresholds\',\'evaluate\',\'analyze\']);a=p.parse_args();torch.set_num_threads(4)\n if a.stage!=\'analyze\':assert torch.cuda.is_available(),\'GPU required for entropy/conditional predictions\'\n globals()[a.stage]()\n'
decile_script=ROOT/'entropy_deciles/entropy_deciles.py'
decile_script.parent.mkdir(parents=True,exist_ok=True)
if decile_script.exists():
    assert decile_script.read_text()==DECILE_SOURCE,'Не смешивай версии decile анализа'
else:
    decile_script.write_text(DECILE_SOURCE)
os.environ['ZR_EXPERIMENTS']=str(EXP)

def run_deciles(stage):
    path=OUT/f'deciles_{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск deciles:',stage,'| лог:',path,flush=True)
    started=time.monotonic()
    with path.open('w') as log:
        process=subprocess.Popen([str(PYTHON),'-u',str(decile_script),stage],cwd=str(EXP),env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30);break
                except subprocess.TimeoutExpired:
                    print(f'{(time.monotonic()-started)/60:.1f} мин | '+' | '.join(path.read_text(errors='replace').splitlines()[-2:]),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    print(path.read_text(errors='replace')[-14000:])
    assert status==0,f'Ошибка: {path}'

run_deciles('thresholds')
run_deciles('evaluate')


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

decile_dir=OUT/'entropy_deciles_v1'
deciles=pd.read_csv(decile_dir/'decile_miou.csv')
macro=pd.read_csv(decile_dir/'macro_decile_miou.csv')
figures=decile_dir/'figures'
figures.mkdir(exist_ok=True)
colors={'entropy':'#64748b','knn_entropy':'#2563eb','Z_entropy':'#ea580c','R_entropy':'#059669','ZR_entropy':'#9333ea','b0':'#111827','b5_full':'#dc2626'}
for domain in ['pool','acdc_all','fog','night','rain','snow']:
    data=deciles[deciles.domain==domain]
    fig,(ax,counts)=plt.subplots(2,1,figsize=(10,8),sharex=True,gridspec_kw={'height_ratios':[3,1]})
    for method,color in colors.items():
        values=data[data.method==method].sort_values('decile')
        ax.plot(values.decile,values.miou,'--' if method in ['b0','b5_full'] else '-o',color=color,label=method,markersize=4)
    support=data[data.method=='b0'].sort_values('decile')
    counts.bar(support.decile,support.valid_pixels,color='#cbd5e1')
    counts.set(xlabel='B0 entropy decile (cuts fitted on clean Cityscapes)',ylabel='Valid pixels',xticks=range(1,11))
    ax.set(title=f'{domain}: conditional mIoU, fixed budget 0.5',ylabel='mIoU, %')
    ax.grid(alpha=.25);ax.legend(fontsize=8,ncol=3)
    fig.tight_layout()
    for extension in ['png','pdf']:fig.savefig(figures/f'{domain}_entropy_deciles.{extension}',dpi=180,bbox_inches='tight')
    plt.show();plt.close(fig)

display(macro[macro.domain.isin(['pool','acdc_all'])])
print('Условные метрики:',decile_dir)


## 8. Скачать результаты и реализацию

Архив: все CSV/JSON/численные результаты, исходники, банкkNN/пороги,8AE и3CatBoost. Исходные датасеты и большой локальный кэш не включены. Эталонные scores.parquet не дублируются: их ID/хеши записаны в протоколе, повторно скачиваются для acceptance.

Для отчёта нужны: `final_miou.csv`, `final_miou_with_gap.csv`, `budget_sweep_v1/curve_metrics.csv` и графики (если sweep выполнен), `bootstrap.csv`, `timing.csv`, `protocol.json`, `acceptance.json`, пороги и сведения о данных. **Не переносим прежний +0.403 в этот эксперимент** — здесь другое правило роутинга.

Если отдельно завершаешь предыдущий оконный опыт, его `acdc_quality.csv` и `acdc_bootstrap.csv` сохраняй в его папке; эти результаты не заменяют друг друга.

In [ ]:
from google.colab import files
archive=ROOT/'ottobiss_z_r_zr_final_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for directory,prefix in [(OUT,'results'),(EXP,'source'),(ART/'ckpt','artifacts/results/ae/ckpt')]:
        for p in directory.rglob('*'):
            if p.is_file() and '__pycache__' not in p.parts:
                z.write(p,prefix+'/'+str(p.relative_to(directory)))
    sweep_script = ROOT/'budget_sweep/budget_sweep.py'
    if sweep_script.is_file():
        z.write(sweep_script, 'budget_sweep/budget_sweep.py')
    decile_script = ROOT/'entropy_deciles/entropy_deciles.py'
    if decile_script.is_file():
        z.write(decile_script, 'entropy_deciles/entropy_deciles.py')
    for kind in ['Z','R','ZR']:
        p=ART/'runs'/(kind+'_ae_b8_catboost')/'model.cbm'
        z.write(p,'artifacts/results/ae/runs/'+kind+'_ae_b8_catboost/model.cbm')
print('Архив:',archive,'|',round(archive.stat().st_size/2**20,1),'МиБ')
files.download(str(archive))
